# E19: vLLM 0.19 MTP + prefix-cache correctness on the S1-60K stack (E19P3)

Does vLLM 0.19's MTP speculative decoding with prefix caching, async scheduling or the fp8 KV cache silently corrupt Qwen3.8-27B
outputs on our exact production stack (S1-60K, LB 34.58)? Engine only, no solver; greedy decoding; about 60 min.

**This notebook's arms:** B (MTP on, prefix off: the MTP-noise calibrator), then K (S1 with bf16 KV). The three notebooks (P1 / P2 / P3) run in parallel on three accounts (arm priority
D, E, B, K, then C, A). `src/e19_compare.py` judges all six arms together from the three bundles.

**Stack (S1 CELL 2 / CELL 5):** vLLM 0.19.0 + transformers 4.57.6 from `mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3`;
FP8 repack, TP4, fp8_e4m3 KV, MTP 3, FLASHINFER when `-lcuda` links, prefix caching, 4,096 batched tokens, memory 0.90,
custom all-reduce off, language-model-only, max_model_len 106,496, 16 seqs, seed 20261005. Here as an OpenAI server from
the kernel's Python, one per arm, killed between arms.

**Probes (identical token ids on every arm):** 12 real Poetiq prompts (S1's render code, xhigh). Serial cold / warm /
cold-repeat greedy 1,024 tokens on 6 of them; a churn pass (24 requests, up to 14 at once, max_tokens 256-2,048,
staggered and refilled arrivals: the batch reorders continuously) compared with the same prompts run alone;
phase-2-style continuations at about 8k and 30k context over the server's own generated tokens; MTP acceptance per
/metrics window.

**Main discriminator: within the arm** (serial A/A noise vs cache hit, vs churn, vs continuation); only large-margin
divergences (top-1 minus top-2 > 1 nat at the first divergent token) and corruption flags count; churn is calibrated
by arms A / B; the token comparison against arm A is informational.

**Verdicts:** PASS / FAIL / NOT_TESTED_BUDGET / INCOMPLETE / INCONCLUSIVE / LAUNCH_FAILED per arm, in
`q27x/e19/e19_report.md`; every raw row in `q27x/e19/e19_rows.jsonl`; bundle `Q27X_E19_BUNDLE.zip`.

**Inputs:** the competition data, the vLLM 0.19 wheelhouse and the FP8 repack model (the same inputs as S1-60K).


In [ ]:
# ==================================================================== CELL 0 — mode + fallback submission (stdlib only)
import os, sys, json, time, random, hashlib, re, shutil, subprocess, threading, traceback, zipfile, math
from pathlib import Path
from datetime import datetime, timezone
import concurrent.futures as cf

NOTEBOOK_T0 = time.perf_counter()
# S1 (CLOCK): Kaggle's 12 h counts from kernel start; R5's cell 0 began 7 s later. Anchor the clock at the kernel
# process start (capped at 10 min in case the value is odd).
try:
    import psutil as _ps
    _since_kernel_start = max(0.0, min(600.0, time.time() - _ps.Process(os.getpid()).create_time()))
except Exception:
    _since_kernel_start = 0.0
NOTEBOOK_T0 -= _since_kernel_start
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RERUN = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true", "yes"}
ARC_ROOT = Path(os.environ.get("Q27X_ARC_ROOT", "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"))
if not ARC_ROOT.exists():
    for _alt in ("/kaggle/input/arc-prize-2026-arc-agi-2",):
        if Path(_alt).exists():
            ARC_ROOT = Path(_alt)
WORKING = Path(os.environ.get("Q27X_WORKING", "/kaggle/working")); WORKING.mkdir(parents=True, exist_ok=True)
OUT = WORKING / "q27x"; OUT.mkdir(parents=True, exist_ok=True)
SUBMISSION_PATH = WORKING / "submission.json"

def _load_json(p):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

test_challenges = _load_json(ARC_ROOT / "arc-agi_test_challenges.json")

def write_json_atomic(path, obj):
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f)
    os.replace(tmp, path)

def identity_submission(challenges):
    return {tid: [{"attempt_1": q["input"], "attempt_2": q["input"]} for q in task["test"]]
            for tid, task in challenges.items()}

write_json_atomic(SUBMISSION_PATH, identity_submission(test_challenges))
print("MODE:", "COMPETITION RERUN (hidden test)" if RERUN else "COMMIT / INTERACTIVE (evaluation)",
      "| fallback submission.json:", len(test_challenges), "tasks | RUN_ID", RUN_ID,
      "| clock anchored", round(_since_kernel_start, 1), "s before cell 0")


In [ ]:
# ==================================================================== CELL 1 — configuration
PROFILE = "RERUN_ALL" if RERUN else os.environ.get("Q27X_PROFILE", "E19P3").strip().upper()
PROFILE_TABLE = {
    # commit run: 12 public evaluation tasks, scored; also measures throughput for the 12-hour projection
    "EVAL12":    {"source": "eval", "n_tasks": 12, "time_budget_sec": 3.0 * 3600, "admit_frac": 0.80, "drain_frac": 0.90},
    # S1 quick commit: 4 eval tasks that closed fastest in R5 iteration 1 -> a valid version in about 1 h, then submit
    "S1COMMIT4": {"source": "eval", "task_list": ["1818057f", "2ba387bc", "bf45cf4b", "58490d8a"],
                  "time_budget_sec": 90 * 60, "admit_frac": 0.80, "drain_frac": 0.90},
    # R5 EVAL120: full evaluation run (120 tasks / 172 inputs) on the hidden-run clock (RERUN_ALL margins)
    "EVAL120":   {"source": "eval", "n_tasks": 120, "time_budget_sec": 12.0 * 3600 - 3 * 60,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
    # E9/E10: PROXY48 (EDA LB-mirror) + the 8 EVAL12 tasks not in it = 56 tasks / 72 test inputs
    "PROXY56":   {"source": "eval", "task_list": ["0934a4d8", "20a9e565", "4c7dc4dd", "89565ca0", "136b0064", "291dc1e1", "16b78196", "271d71e2", "28a6681f", "2b83f449", "2c181942", "31f7f899", "35ab12c3", "3dc255db", "409aa875", "4c416de3", "64efde09", "6e453dd6", "7491f3cf", "7b0280bc", "7b80bb43", "7c66cb00", "1818057f", "332f06d7", "53fb4810", "7666fa5d", "da515329", "8f3a5a89", "db695cfb", "97d7923e", "de809cff", "c7f57c3e", "cb2d8a2c", "58490d8a", "dbff022c", "5961cc34", "80a900e0", "269e22fb", "8b9c3697", "e376de54", "b99e7126", "c4d067a0", "135a2760", "4a21e3da", "71e489b6", "800d221b", "dd6b8c4b", "36a08778", "21897d95", "247ef758", "4c3d4a41", "6e4f6532", "88bcf3b4", "898e7135", "8b7bacbf", "b6f77b65"], "time_budget_sec": 6.0 * 3600,
                  "admit_frac": 0.80, "drain_frac": 0.90},
    # E19 (2026-10-27): vLLM 0.19 MTP / prefix-cache / async / KV-dtype correctness probes, 60 min per account;
    # e19_arms = [arm, minutes reserved for it while an earlier arm runs]; arm priority D, E, B, K, then C, A (each
    # account runs one high-priority MTP arm first); the 12 G4B eval tasks only feed real Poetiq prompts
    "E19P1":     {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd", "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"], "time_budget_sec": 60 * 60,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 3 * 60, "verify_margin_sec": 1 * 60,
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "e19_arms": [["D", 0], ["C", 22]]},
    "E19P2":     {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd", "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"], "time_budget_sec": 60 * 60,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 3 * 60, "verify_margin_sec": 1 * 60,
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "e19_arms": [["E", 0], ["A", 22]]},
    "E19P3":     {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd", "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"], "time_budget_sec": 60 * 60,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 3 * 60, "verify_margin_sec": 1 * 60,
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "e19_arms": [["B", 0], ["K", 18]]},
    # all six arms on one account (local harness; about 2.5 h on a GPU)
    "E19ALL":    {"source": "eval", "task_list": ["136b0064", "898e7135", "135a2760", "b5ca7ac4", "4c3d4a41", "62593bfd", "247ef758", "bf45cf4b", "f931b4a8", "65b59efc", "9385bd28", "cbebaa4b"], "time_budget_sec": 240 * 60,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 3 * 60, "verify_margin_sec": 1 * 60,
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60, "e19_arms": [["A", 0], ["B", 0], ["C", 0], ["D", 0], ["E", 0], ["K", 0]]},
    # hidden test: every task; Kaggle's limit is 12 h including install + model load
    # v1.4.3: stop 10 min before 12 h (user); hard stop 6 min before the budget end (outputs take seconds after it)
    "RERUN_ALL": {"source": "test", "n_tasks": None, "time_budget_sec": 12.0 * 3600 - 3 * 60,
                  "admit_frac": 0.90, "drain_frac": 0.94, "hard_margin_sec": 2 * 60, "verify_margin_sec": 1 * 60,
                  # v1.4.4: admit until the end (NVARC style); drain 8 min before the hard stop, admit 2 min before drain
                  "drain_margin_sec": 8 * 60, "admit_margin_sec": 2 * 60},
}
if PROFILE not in PROFILE_TABLE:
    raise ValueError(f"unknown profile {PROFILE}; choose {list(PROFILE_TABLE)}")
P = PROFILE_TABLE[PROFILE]
SEED = 20261005
TASK_SAMPLE_SEED = 20261005

MODEL_PATH = Path(os.environ.get("Q27X_MODEL", "/kaggle/input/models/foysalemonshanto/qwen3-8-27b-fp8-repacked-v1/pytorch/hf-fp8/1"))
WHEELHOUSE = Path(os.environ.get("Q27X_WHEELHOUSE", "/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3"))

# ---- generation budget (Kaggle discussion 745787: Qwen3.8 needs xhigh; median ~64k tokens; forced commit ~49k)
REASONING_EFFORT = "xhigh"
THINK_BUDGET = int(os.environ.get("Q27X_THINK_BUDGET", 60000))   # S1-60K: LB test of 60,000 (R5/E9: 77,824; floor 49,152)      # v1.2: NguyenThanhNhan forced the commit after ~49k thinking tokens
ANSWER_MAX_TOKENS = int(os.environ.get("Q27X_ANSWER_MAX", 8192))   # v1.4: grid-only answer; 49,152 + 8,192 = 57,344 = russcore's max_tokens
DRAIN_ANSWER_MAX_TOKENS = 8192                                      # v1.1 (F11): a 30x30 grid alone is ~2.7k tokens
STOP_THINKING_TEXT = "\n\nConsidering the limited time by the user, I have to give the solution based on the thinking directly now.\n</think>\n\n"
MAX_MODEL_LEN = int(os.environ.get("Q27X_MAX_MODEL_LEN", 106496))   # E9/E10: prompt (<= ~20k) + 77,824 + 8,192 fits; was 81,920   # v1.2: native context is 262k; fp8 KV makes room
CONTEXT_SAFETY = 256
MIN_THINK_BUDGET = 4096                  # v1.2: a long prompt gets a smaller thinking budget instead of being skipped
PROMPT_TOKEN_LIMIT = MAX_MODEL_LEN - ANSWER_MAX_TOKENS - CONTEXT_SAFETY - MIN_THINK_BUDGET
def phase1_cap(prompt_len):
    return max(0, min(THINK_BUDGET, MAX_MODEL_LEN - prompt_len - ANSWER_MAX_TOKENS - CONTEXT_SAFETY))
assert phase1_cap(MAX_MODEL_LEN) == 0 and phase1_cap(100) == min(THINK_BUDGET, MAX_MODEL_LEN - 100 - ANSWER_MAX_TOKENS - CONTEXT_SAFETY)
if (MAX_MODEL_LEN, THINK_BUDGET, ANSWER_MAX_TOKENS) == (106496, 77824, 8192):   # E9/E10
    assert phase1_cap(12800) == 77824 and phase1_cap(20000) == 77824 and phase1_cap(30000) == 68048
if (MAX_MODEL_LEN, THINK_BUDGET, ANSWER_MAX_TOKENS) == (81920, 49152, 8192):
    assert phase1_cap(24320) == 49152 and phase1_cap(30000) == 43472  # every real prompt (≤ ~9k tokens) keeps the full budget
SAMPLING = {"temperature": float(os.environ.get("Q27X_TEMPERATURE", 0.6)), "top_p": 0.95, "top_k": 20}   # v1.4: russcore LB-10.14 sampling

# ---- engine (PIV/V3.1 C1 settings; context, concurrency and MTP changed)
MAX_NUM_SEQS = int(os.environ.get("Q27X_MAX_NUM_SEQS", 16))   # upper bound; actual slots = KV tokens // EXPECTED_LIVE_TOKENS_PER_SEQ
MAX_NUM_BATCHED_TOKENS = 4096   # v1.3.5: = discussion max-prefill-tokens 4096; 32768 ran out of memory in run 2 (MTP drafter)
GPU_MEMORY_UTILIZATION = float(os.environ.get("Q27X_GPU_MEM", 0.90))   # v1.4.1: kept at 0.90 so prefix caching is the only change
ENABLE_PREFIX_CACHING = os.environ.get("Q27X_PREFIX_CACHE", "1") == "1"   # v1.4.1: first engine attempt (mamba 'align')
SPECULATIVE_CONFIG = {"method": "mtp", "num_speculative_tokens": int(os.environ.get("Q27X_SPEC_TOKENS", 3))}   # v1.2: vLLM Qwen3.8 recipe uses 3
KV_CACHE_DTYPE = os.environ.get("Q27X_KV_DTYPE", "fp8_e4m3")   # v1.3.4: back to the discussion recipe; run 1's crash was
# FlashInfer's JIT link step (`ld: cannot find -lcuda`), fixed in cell 5 (libcuda shim + link probe); fp8 KV itself loaded
# fine (312,000 KV tokens). Set Q27X_KV_DTYPE=auto for bf16 KV.
if os.environ.get("Q27X_NO_SPEC") == "1":
    SPECULATIVE_CONFIG = None
EXPECTED_LIVE_TOKENS_PER_SEQ = 43000      # a running task holds on average about half of its final ~60k length
# v1.3.5 correction: cell 5 divides the whole KV pool (run 2: 781 blocks × 1,600 = 1,249,600 tokens) by this. The pool is
# shared by 1 attention + 3 GDN groups; vLLM's "312,000 tokens" log line is pool // 4. A 30k-token task needs 19 attention
# + 12 GDN-state blocks (3 groups × (1 + 3 MTP)), so 16 tasks need ~496 of 781 blocks -> 16 slots (preemptions reported)

# ---- verification (fresh process per candidate: train first, then test)
VERIFY_TIMEOUT_SEC = 20.0
VERIFY_WORKERS = 4

# ---- time
TOTAL_BUDGET_SEC = float(os.environ.get("Q27X_TIME_BUDGET", P["time_budget_sec"]))
# v1.3: deadlines derived in a guaranteed order (v1.2 had DRAIN after HARD_STOP for EVAL12, so the drain never ran)
VERIFY_DEADLINE = NOTEBOOK_T0 + TOTAL_BUDGET_SEC - P.get("verify_margin_sec", 8 * 60)   # reporting only since v1.4
HARD_STOP = NOTEBOOK_T0 + TOTAL_BUDGET_SEC - P.get("hard_margin_sec", 25 * 60)        # abort everything, write outputs
DRAIN_DEADLINE = min(NOTEBOOK_T0 + TOTAL_BUDGET_SEC * P["drain_frac"], HARD_STOP - 8 * 60)    # force-close thinking
ADMIT_DEADLINE = min(NOTEBOOK_T0 + TOTAL_BUDGET_SEC * P["admit_frac"], DRAIN_DEADLINE - 7 * 60)  # no new task after this
if "drain_margin_sec" in P:   # v1.4.4 (RERUN): fixed offsets from the hard stop, so the slots stay full until the drain
    DRAIN_DEADLINE = HARD_STOP - P["drain_margin_sec"]
    ADMIT_DEADLINE = DRAIN_DEADLINE - P["admit_margin_sec"]
assert NOTEBOOK_T0 < ADMIT_DEADLINE < DRAIN_DEADLINE < HARD_STOP < VERIFY_DEADLINE < NOTEBOOK_T0 + TOTAL_BUDGET_SEC

# v1.3.5 (run 2 OOM): CUDA-graph memory is taken out of the KV budget (vLLM 0.19 envs.py 1654, v1/worker/gpu_worker.py
# 393-420); run 2 captured 0.61 GiB of graphs beyond it. Set here, before cell 5 spawns the vLLM workers (they inherit it).
os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = "1"
CONFIG = {k: globals()[k] for k in ["PROFILE", "RERUN", "SEED", "REASONING_EFFORT", "THINK_BUDGET", "ANSWER_MAX_TOKENS",
          "DRAIN_ANSWER_MAX_TOKENS", "MAX_MODEL_LEN", "PROMPT_TOKEN_LIMIT", "SAMPLING", "MAX_NUM_SEQS",
          "MAX_NUM_BATCHED_TOKENS", "GPU_MEMORY_UTILIZATION", "ENABLE_PREFIX_CACHING", "SPECULATIVE_CONFIG", "KV_CACHE_DTYPE", "VERIFY_TIMEOUT_SEC",
          "TOTAL_BUDGET_SEC", "MIN_THINK_BUDGET"]}
CONFIG["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"] = os.environ["VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS"]
CONFIG["profile_row"] = P
print(json.dumps(CONFIG, indent=1, default=str))

os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
os.environ["HF_HUB_OFFLINE"] = "1"; os.environ["TRANSFORMERS_OFFLINE"] = "1"


# ---- E10 Poetiq donor configurations; source: poetiq-ai/poetiq-arc-agi-solver/config.py
# Original 86K context, GPU recipe, deadlines and original puzzle order above are unchanged.
POETIQ_NUM_EXPERTS = int(os.environ.get('E10_POETIQ_EXPERTS','1'))
POETIQ_MAX_ITERATIONS = int(os.environ.get('E10_POETIQ_MAX_ITERATIONS','10'))
POETIQ_MAX_SOLUTIONS = int(os.environ.get('E10_POETIQ_MAX_SOLUTIONS','5'))
POETIQ_SOLVER_TEMPERATURE = float(os.environ.get('E10_POETIQ_TEMPERATURE','1.0'))
POETIQ_SOLVER_VARIANTS = tuple(int(v) for v in os.environ.get('E10_POETIQ_PROMPT_VARIANTS','1').split(','))
POETIQ_SELECTION_PROB = float(os.environ.get('E10_POETIQ_SELECTION_PROBABILITY','1.0'))
POETIQ_SEED = int(os.environ.get('E10_POETIQ_SEED','0'))
POETIQ_SHUFFLE_EXAMPLES = os.environ.get('E10_POETIQ_SHUFFLE','1')=='1'
POETIQ_IMPROVING_ORDER = os.environ.get('E10_POETIQ_IMPROVING_ORDER','1')=='1'
POETIQ_USE_NEW_VOTING = os.environ.get('E10_POETIQ_NEW_VOTING','1')=='1'
POETIQ_COUNT_FAILED_MATCHES = os.environ.get('E10_POETIQ_COUNT_FAILED_MATCHES','1')=='1'
POETIQ_ITERS_TIEBREAK = os.environ.get('E10_POETIQ_ITERS_TIEBREAK','0')=='1'
POETIQ_LOW_TO_HIGH_ITERS = os.environ.get('E10_POETIQ_LOW_TO_HIGH_ITERS','0')=='1'
POETIQ_SANDBOX_TIMEOUT = float(os.environ.get('E10_POETIQ_SANDBOX_TIMEOUT','5'))
POETIQ_SANDBOX_WORKERS = int(os.environ.get('E10_POETIQ_SANDBOX_WORKERS','4'))
# S1: KV admission gate (no new request above this vLLM KV usage unless fewer than MIN_ACTIVE are running)
E10_KV_ADMIT_MAX = float(os.environ.get('E10_KV_ADMIT_MAX','0.95'))
E10_KV_GATE_MIN_ACTIVE = int(os.environ.get('E10_KV_GATE_MIN_ACTIVE','12'))
assert POETIQ_NUM_EXPERTS >= 1 and POETIQ_MAX_ITERATIONS >= 1
assert POETIQ_SOLVER_VARIANTS and all(v in (1,2,3) for v in POETIQ_SOLVER_VARIANTS)
assert POETIQ_MAX_SOLUTIONS >= 1 and 0 <= POETIQ_SELECTION_PROB <= 1
CONFIG['poetiq']={k:globals()[k] for k in (
    'POETIQ_NUM_EXPERTS','POETIQ_MAX_ITERATIONS','POETIQ_MAX_SOLUTIONS',
    'POETIQ_SOLVER_TEMPERATURE','POETIQ_SOLVER_VARIANTS','POETIQ_SELECTION_PROB',
    'POETIQ_SANDBOX_TIMEOUT','POETIQ_SANDBOX_WORKERS','POETIQ_USE_NEW_VOTING')}


# ---- E19 (2026-10-27): does vLLM 0.19's MTP + prefix caching (and fp8 KV) silently corrupt greedy outputs on the S1
# stack? Engine only, no solver. Commit mode only: the competition rerun keeps CELL 0's identity submission.
E19_ON = bool(P.get("e19_arms")) and not RERUN
if E19_ON:
    assert SPECULATIVE_CONFIG == {"method": "mtp", "num_speculative_tokens": 3} and KV_CACHE_DTYPE == "fp8_e4m3" \
        and ENABLE_PREFIX_CACHING and MAX_NUM_SEQS == 16 and MAX_MODEL_LEN == 106496, "E19 needs S1's engine settings"
# each arm changes only its own switches; every other flag = S1's ENGINE_CONFIG (CELL 5)
E19_ARMS = {
    "D": {"spec": 3, "prefix": True, "kv": "fp8_e4m3", "what": "S1 exactly: MTP 3 + prefix caching ON, fp8 KV"},
    "E": {"spec": 3, "prefix": True, "kv": "fp8_e4m3", "async": False, "what": "S1 + --no-async-scheduling"},
    "B": {"spec": 3, "prefix": False, "kv": "fp8_e4m3", "what": "MTP 3, prefix caching OFF"},
    "C": {"spec": None, "prefix": True, "kv": "fp8_e4m3", "what": "MTP OFF, prefix caching ON"},
    "A": {"spec": None, "prefix": False, "kv": "fp8_e4m3", "what": "reference: MTP OFF, prefix caching OFF"},
    "K": {"spec": 3, "prefix": True, "kv": "auto", "what": "S1 with bf16 KV (--kv-cache-dtype auto)"},
}
E19_PLAN = {"n_serial": 6, "max_new": 1024, "g_pre": 512, "cmp": 512, "block": 1600, "backoff": 256,
            # churn: the 6 serial prompts x 4, up to 14 at once, max_tokens staggered, arrivals staggered + refilled
            "churn_reps": 4, "churn_conc": 14, "churn_max_tokens": [256, 512, 768, 1024, 1280, 1536, 1792, 2048],
            "churn_stagger_sec": float(os.environ.get("E19_CHURN_STAGGER_SEC", 0.5)),
            "long_conc": 6, "long": [["long8k", 8192, "other"], ["long30k", 30720, "serial"]]}
E19_LAUNCH_MIN_SEC = float(os.environ.get("E19_LAUNCH_MIN_SEC", 14 * 60))   # no launch with less time before the arm deadline
E19_MIN_SERIAL_SEC = float(os.environ.get("E19_MIN_SERIAL_SEC", 150))       # one serial prompt (3 x 1,024 greedy tokens)
E19_MIN_CHURN_SEC = float(os.environ.get("E19_MIN_CHURN_SEC", 150))         # the churn stage (24 requests, 14 at once)
E19_MIN_LONG_SEC = float(os.environ.get("E19_MIN_LONG_SEC", 240))           # one long-context stage
E19_REPORT_RESERVE_SEC = float(os.environ.get("E19_REPORT_RESERVE_SEC", 90))
E19_SERIAL_IDX, E19_LONG, E19_CHURN_JOBS, E19_CUR = [], [], [], None
CONFIG["e19"] = {"on": E19_ON, "arms": P.get("e19_arms"), "plan": E19_PLAN}
print("E19:", json.dumps(CONFIG["e19"]))


In [ ]:
# ==================================================================== CELL 2 — offline install (PIV/V3.1 parity block) + tokenizer
import importlib, importlib.metadata
def pkg_version(name):
    try: return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: return None

EXPECTED_RUNTIME_PACKAGES = {"vllm": "0.19.0", "transformers": "4.57.6", "tokenizers": "0.22.2", "huggingface-hub": "0.36.2"}
_loaded = {"vllm": "vllm", "transformers": "transformers", "tokenizers": "tokenizers", "huggingface-hub": "huggingface_hub"}
before_stack = {n: pkg_version(n) for n in EXPECTED_RUNTIME_PACKAGES}

def offline_pip_install(requirements):
    if not requirements: return
    if not WHEELHOUSE.exists():
        raise FileNotFoundError(f"wheelhouse not attached: {WHEELHOUSE}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--no-cache-dir",
                           "--find-links", str(WHEELHOUSE), *requirements])

# S1: read the model files into the page cache in the background while pip runs (read-only, best effort)
if os.environ.get("Q27X_SKIP_INSTALL") != "1":
    try:
        import shlex as _shlex
        _wfiles = sorted(str(p) for p in MODEL_PATH.glob("*.safetensors"))
        if _wfiles:
            subprocess.Popen(["bash", "-c", "cat " + " ".join(_shlex.quote(p) for p in _wfiles) + " > /dev/null"],
                             stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            print("weight prefetch started:", len(_wfiles), "files")
    except Exception as _e:
        print("weight prefetch not started:", repr(_e)[:200])

if os.environ.get("Q27X_SKIP_INSTALL") != "1":     # Q27X_SKIP_INSTALL is for the local harness only
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError(f"Python {sys.version.split()[0]}: the cp312 wheelhouse needs the pinned Python 3.12 Kaggle image")
    for _pkg, _mod in _loaded.items():
        if before_stack.get(_pkg) != EXPECTED_RUNTIME_PACKAGES[_pkg] and _mod in sys.modules:
            raise RuntimeError(f"{_pkg} already imported at {before_stack.get(_pkg)!r}; restart the session before Run All.")
    if before_stack["vllm"] != EXPECTED_RUNTIME_PACKAGES["vllm"]:
        offline_pip_install(["vllm==0.19.0"])
    else:
        offline_pip_install([f"{p}=={EXPECTED_RUNTIME_PACKAGES[p]}" for p in ("transformers", "tokenizers", "huggingface-hub")
                             if before_stack[p] != EXPECTED_RUNTIME_PACKAGES[p]])
    after_stack = {n: pkg_version(n) for n in EXPECTED_RUNTIME_PACKAGES}
    for _p, _want in EXPECTED_RUNTIME_PACKAGES.items():
        if after_stack.get(_p) != _want:
            raise RuntimeError(f"runtime parity failure: {_p}={after_stack.get(_p)!r}, expected {_want!r}")
    print("runtime parity PASS:", after_stack)

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(str(MODEL_PATH), trust_remote_code=False)
_model_cfg = _load_json(MODEL_PATH / "config.json")
assert _model_cfg.get("model_type") == "qwen3_5", _model_cfg.get("model_type")
assert (_model_cfg.get("quantization_config") or {}).get("quant_method") == "fp8"
THINK_CLOSE_IDS = tokenizer.encode("</think>", add_special_tokens=False)
assert len(THINK_CLOSE_IDS) == 1, THINK_CLOSE_IDS
THINK_CLOSE_ID = THINK_CLOSE_IDS[0]
STOP_IDS = tokenizer.encode(STOP_THINKING_TEXT, add_special_tokens=False)
assert THINK_CLOSE_ID in STOP_IDS
_gen_cfg = _load_json(MODEL_PATH / "generation_config.json") if (MODEL_PATH / "generation_config.json").exists() else {}
_eos = _gen_cfg.get("eos_token_id", [])
EOS_IDS = set(_eos if isinstance(_eos, list) else [_eos])     # v1.1 (F8): stripped before a forced close
print("tokenizer OK | </think> id", THINK_CLOSE_ID, "| stop text", len(STOP_IDS), "tokens")


In [ ]:
# E10: original task-order and evaluation profiles, but one Poetiq program per task/expert/iteration.
if P['source']=='eval':
    _ch=_load_json(ARC_ROOT/'arc-agi_evaluation_challenges.json')
    _sol=_load_json(ARC_ROOT/'arc-agi_evaluation_solutions.json')
    if P.get('task_list'):
        assert all(t in _ch for t in P['task_list']), 'Evaluation task ID missing'
        TASK_IDS=sorted(P['task_list'])
    else:
        TASK_IDS=sorted(random.Random(TASK_SAMPLE_SEED).sample(sorted(_ch),P['n_tasks']))
    CHALLENGES={t:_ch[t] for t in TASK_IDS}
    SOLUTIONS={t:_sol[t] for t in TASK_IDS}
else:
    CHALLENGES=dict(test_challenges)
    TASK_IDS=sorted(CHALLENGES) # no Codex ordering
    SOLUTIONS=None
N_HIDDEN_UNITS=sum(len(t['test']) for t in test_challenges.values())
print('E10 Poetiq:',len(TASK_IDS),'tasks,',sum(len(CHALLENGES[t]['test']) for t in TASK_IDS),
      'queries | experts',POETIQ_NUM_EXPERTS,'iterations per expert',POETIQ_MAX_ITERATIONS)


In [ ]:
# E10 Poetiq donor adaptation: task formatting, code extraction, sandbox, feedback, voting.
# Donor: https://github.com/poetiq-ai/poetiq-arc-agi-solver (MIT)
# Donor originals: arc_agi/solve_coding.py, solve_parallel_coding.py, sandbox.py, prompts.py
import numpy as np
import json, re, sys, subprocess, tempfile, textwrap, math, time, os, string
from pathlib import Path

# Donor prompt variants adapted solely to the local Q27 chat interface.
# The task wording, Python transform contract, and refinement feedback follow Poetiq.
POETIQ_SOLVER_PROMPTS = {
  1: '''You are an expert in solving Abstract Reasoning Corpus (ARC) tasks by writing Python code. Your goal is to analyze input-output examples and create a 'transform' function that correctly transforms any given input grid into the corresponding output grid.
Analyze the examples: objects, colors, spatial relations, shapes, lines, symmetries, sizes. Formulate one consistent rule, prioritize simple general rules. Implement a Python function `transform(grid: np.ndarray) -> np.ndarray` with NumPy or standard libraries. Return a single complete markdown ```python code block containing the function; include needed imports. No __main__ block. The program must work on all training inputs and all test inputs. Do not hardcode specific test outputs.
PROBLEM:\n$$problem$$''',
  2: '''You are a world-class expert in solving ARC tasks and an expert Python coder. Analyze every example carefully, then hypothesize a SINGLE consistent rule applicable to all examples. Consider colors, connected components, rotations, reflections, object motion, addition/removal, spatial patterns and symmetries. Test and refine the hypothesis to match all example outputs. Implement `def transform(grid: np.ndarray) -> np.ndarray` using NumPy (cv2 and standard libraries may be used if available). Your answer must contain a single valid markdown ```python code block defining the function and any imports. Explain briefly before the code. Do not output a __main__ block.
PROBLEM:\n$$problem$$''',
  3: '''You are a world-class ARC reasoning researcher and master Python coder. Inspect the dimensions, positions, colors, objects, neighborhoods and geometry in all demonstration input/output pairs. Generate possible hypotheses; use the simplest rule explaining all pairs. Translate it into executable, documented Python code using NumPy and optionally cv2/standard libraries. Handle edge cases, unexpected shapes and colors. Return one markdown ```python code block containing imports and `def transform(grid: np.ndarray) -> np.ndarray`, plus a short explanation. Do not include a __main__ block. The same rule must generalize to each Challenge input.
PROBLEM:\n$$problem$$'''
}
POETIQ_FEEDBACK_PROMPT = '''**EXISTING PARTIAL/INCORRECT SOLUTIONS:**\nFollowing are some of the best, though not completely correct, solutions so far. For each solution, code, training-example feedback, and a score from 0 to 1 are provided. Study the code and the feedback and produce a NEW complete Python `transform` solution fixing all the issues. Follow the exact output format.\n$$feedback$$'''

def poetiq_diagram(g):
    return '\n'.join(' '.join(str(int(v)) for v in row) for row in g)

def poetiq_format_problem(task, shuffle=False, seed=0):
    train = list(task['train'])
    if shuffle and len(train) > 1:
        perm = np.random.default_rng(seed).permutation(len(train))
        train = [train[int(i)] for i in perm]
    chunks=[]
    for i, item in enumerate(train,1):
        chunks.append(f'Example #{i}\nInput:\n<Diagram>\n{poetiq_diagram(item["input"])}\n</Diagram>\nOutput:\n<Diagram>\n{poetiq_diagram(item["output"])}\n</Diagram>')
    for i,item in enumerate(task['test'],1):
        chunks.append(f'Challenge #{i}\nInput:\n<Diagram>\n{poetiq_diagram(item["input"])}\n</Diagram>')
    return '\n\n'.join(chunks)

def poetiq_parse_code(response):
    # S1 (F1): the LAST ```python block that parses and defines transform(); R5 took the first block, so a short
    # illustrative snippet written before the real solution was the one executed. Falls back to the R5 choice.
    import ast as _ast
    blocks=re.findall(r'```python\s*(.*?)```',response or '',re.DOTALL|re.IGNORECASE)
    for b in reversed(blocks):
        b=b.strip()
        try:
            tree=_ast.parse(b)
        except (SyntaxError,ValueError):
            continue
        if any(isinstance(n,_ast.FunctionDef) and n.name=='transform' for n in tree.body):
            return b
    return blocks[0].strip() if blocks else None

def poetiq_create_examples(solutions, max_examples=5, improving_order=True):
    if not solutions: return ''
    chosen=sorted(solutions,key=lambda z:z['score'], reverse=True)[:max_examples]
    if improving_order: chosen.reverse()
    return '\n\n'.join(f'<solution_{k}>\n<solution_code>\n```python\n{s["code"]}\n```\n</solution_code>\n<solution_evaluation>\n{s["feedback"]}\n</solution_evaluation>\n<solution_score>\n{s["score"]:.2f}\n</solution_score>\n</solution_{k}>' for k,s in enumerate(chosen,1))

def poetiq_build_message(task,iteration,expert_idx,solutions,variant=1,selection_probability=1.,max_solutions=5,improving_order=True,shuffle=True,seed=0):
    # Poetiq's seeded example shuffling and best-history feedback.
    problem=poetiq_format_problem(task, shuffle, seed+iteration)
    msg=POETIQ_SOLVER_PROMPTS[int(variant)].replace('$$problem$$',problem)
    if solutions and max_solutions > 0:
        rng=np.random.default_rng(seed+100000*expert_idx+iteration)
        subset=[s for s in solutions if rng.random()<selection_probability]
        if subset:
            msg+='\n\n'+POETIQ_FEEDBACK_PROMPT.replace('$$feedback$$',poetiq_create_examples(subset,max_solutions,improving_order))
    return msg

def poetiq_normalize_grid(g):
    """Lossless ARC palette normalization, rejecting booleans and fractions."""
    if not isinstance(g, list) or not 1 <= len(g) <= 30:
        return None
    if not all(isinstance(row,list) and 1 <= len(row) <= 30 for row in g):
        return None
    if len({len(row) for row in g}) != 1:
        return None
    normalized=[]
    for row in g:
        rr=[]
        for v in row:
            if isinstance(v, (bool, np.bool_)):
                return None
            if isinstance(v,(int,np.integer)):
                value=int(v)
            elif isinstance(v,(float,np.floating)) and math.isfinite(float(v)) and float(v).is_integer():
                value=int(v)
            else:
                return None
            if not 0 <= value <= 9:
                return None
            rr.append(value)
        normalized.append(rr)
    return normalized


def poetiq_valid_grid(g):
    return poetiq_normalize_grid(g) is not None


def poetiq_grid_from_output(text):
    try:
        return poetiq_normalize_grid(json.loads(text))
    except (TypeError,ValueError,OverflowError):
        return None


# Poetiq subprocess execution adapted for Kaggle Docker without unshare.
# UID separation + parent limits + no-new-privs + seccomp are enforced.
# These do not create a mount namespace. World-readable host files are NOT
# guaranteed inaccessible; see the README's explicit residual risk.
_POETIQ_TRUSTED_RUNNER = '"""Trusted Poetiq runner for Kaggle\'s namespace-restricted Docker containers.\n\nNot a mount namespace. It provides UID separation and syscall/resource limits;\nworld-readable host paths are NOT guaranteed unreadable without a mount or LSM.\n"""\nimport sys, os, json, ctypes, errno, resource, traceback\n# Import numerical/scientific libraries before syscall filtering.\nimport numpy as np\ntry:\n    import scipy\nexcept ImportError:\n    scipy = None\ntry:\n    import cv2\nexcept ImportError:\n    cv2 = None\n\n_RESULT_MARKER = \'__E10_PROGRAM_RESULT__\'\n\n\ndef drop_privileges_and_limits():\n    # Trusted code runs before compiling or executing model-generated Python.\n    if os.geteuid() != 0:\n        raise RuntimeError(\'Root is required for irreversible UID/GID drop\')\n    os.setgroups([])\n    os.setgid(65534)\n    os.setuid(65534)\n    cpu = max(2,min(120,int(os.environ.get(\'E10_CHILD_CPU\',\'10\'))))\n    mem = 4096*1024*1024\n    resource.setrlimit(resource.RLIMIT_CPU,(cpu,cpu+1))\n    resource.setrlimit(resource.RLIMIT_AS,(mem,mem))\n    resource.setrlimit(resource.RLIMIT_FSIZE,(1024*1024,1024*1024))\n    resource.setrlimit(resource.RLIMIT_CORE,(0,0))\n    resource.setrlimit(resource.RLIMIT_NOFILE,(64,64))\n    resource.setrlimit(resource.RLIMIT_NPROC,(64,64))\n\n\ndef harden_process():\n    libc = ctypes.CDLL(None, use_errno=True)\n    if libc.prctl(38, 1, 0, 0, 0) != 0:\n        raise RuntimeError(\'PR_SET_NO_NEW_PRIVS failed\')\n    lib = ctypes.CDLL(\'libseccomp.so.2\', use_errno=True)\n    lib.seccomp_init.argtypes = [ctypes.c_uint32]\n    lib.seccomp_init.restype = ctypes.c_void_p\n    lib.seccomp_syscall_resolve_name.argtypes = [ctypes.c_char_p]\n    lib.seccomp_syscall_resolve_name.restype = ctypes.c_int\n    lib.seccomp_rule_add.argtypes = [ctypes.c_void_p, ctypes.c_uint32, ctypes.c_int, ctypes.c_uint]\n    lib.seccomp_rule_add.restype = ctypes.c_int\n    lib.seccomp_load.argtypes = [ctypes.c_void_p]\n    lib.seccomp_load.restype = ctypes.c_int\n    lib.seccomp_release.argtypes = [ctypes.c_void_p]\n    lib.seccomp_release.restype = None\n    ctx = lib.seccomp_init(0x7fff0000)\n    if not ctx:\n        raise RuntimeError(\'seccomp_init failed\')\n    try:\n        # Blocking exec/spawn/network/ptrace/mount remains compatible with\n        # numpy/scipy/cv2. Do not block clone(thread) indiscriminately.\n        blocked = (\n            \'execve\',\'execveat\',\'fork\',\'vfork\',\n            \'socket\',\'socketpair\',\'connect\',\'bind\',\'listen\',\'accept\',\'accept4\',\n            \'mount\',\'umount2\',\'pivot_root\',\'chroot\',\'unshare\',\'setns\',\n            \'ptrace\',\'process_vm_readv\',\'process_vm_writev\',\'open_by_handle_at\',\n            \'reboot\',\'bpf\',\'keyctl\',\'setuid\',\'setgid\',\'setreuid\',\'setregid\',\n            \'setresuid\',\'setresgid\',\n        )\n        for name in blocked:\n            nr = lib.seccomp_syscall_resolve_name(name.encode())\n            if nr >= 0:\n                if lib.seccomp_rule_add(ctx, 0x00050000 | errno.EPERM, nr, 0):\n                    raise RuntimeError(\'seccomp rule failed: \' + name)\n        # glibc falls back from clone3 to clone only on ENOSYS, not EPERM.\n        # Keep process creation blocked by the existing clone/CLONE_THREAD rule.\n        clone3_nr = lib.seccomp_syscall_resolve_name(b\'clone3\')\n        if clone3_nr >= 0:\n            rc = lib.seccomp_rule_add(ctx, 0x00050000 | errno.ENOSYS, clone3_nr, 0)\n            if rc:\n                raise RuntimeError(\'seccomp clone3 restriction failed: \' + str(rc))\n        # clone() flags: require CLONE_THREAD when creating a thread.\n        # This blocks ordinary child processes without disabling numerical threads.\n        clone_nr = lib.seccomp_syscall_resolve_name(b\'clone\')\n        if clone_nr >= 0:\n            class ArgCmp(ctypes.Structure):\n                _fields_=[(\'arg\',ctypes.c_uint),(\'op\',ctypes.c_uint),\n                          (\'datum_a\',ctypes.c_uint64),(\'datum_b\',ctypes.c_uint64)]\n            # seccomp_rule_add_array(ctx, action, syscall, n, &cmp)\n            lib.seccomp_rule_add_array.argtypes=[ctypes.c_void_p,ctypes.c_uint32,ctypes.c_int,\n                                                   ctypes.c_uint,ctypes.POINTER(ArgCmp)]\n            lib.seccomp_rule_add_array.restype=ctypes.c_int\n            # SCMP_CMP_MASKED_EQ=7; deny clone if CLONE_THREAD bit is zero.\n            cmp=ArgCmp(0,7,0x10000,0)\n            rc=lib.seccomp_rule_add_array(ctx,0x00050000|errno.EPERM,clone_nr,1,ctypes.byref(cmp))\n            if rc:\n                raise RuntimeError(\'seccomp clone restriction failed: \'+str(rc))\n        rc=lib.seccomp_load(ctx)\n        if rc:\n            raise RuntimeError(\'seccomp_load failed \'+str(rc))\n    finally:\n        lib.seccomp_release(ctx)\n\n\ndef main():\n    raw = json.load(sys.stdin)\n    program = raw[\'code\']\n    grid = raw[\'input\']\n    if not isinstance(program, str) or len(program)>250000:\n        raise ValueError(\'invalid code length\')\n    drop_privileges_and_limits()\n    harden_process()\n    namespace = {\'__name__\': \'poetiq_candidate\', \'__builtins__\': __builtins__, \'np\': np}\n    try:\n        exec(compile(program, \'<poetiq_candidate>\', \'exec\'), namespace)\n        result=namespace[\'transform\'](np.array(grid, dtype=int))\n        if hasattr(result,\'tolist\'):\n            result=result.tolist()\n        payload={\'ok\':True,\'result\':result}\n    except BaseException as ex:\n        payload={\'ok\':False,\'error\':(type(ex).__name__+\': \'+str(ex))[:1200]}\n    try:\n        print(_RESULT_MARKER + json.dumps(payload,separators=(\',\',\':\')),flush=True)\n    except BaseException as ex:\n        print(_RESULT_MARKER + json.dumps({\'ok\':False,\'error\':\'JSON output error: \'+str(ex)[:500]}),flush=True)\n\nif __name__==\'__main__\':\n    try:\n        main()\n    except BaseException as ex:\n        sys.stderr.write(\'E10_RUNNER_SETUP_ERROR: \'+repr(ex)+\'\\n\')\n        sys.exit(110)\n'



def poetiq_sandbox_run(code, grid, timeout_s=5.0):
    import signal
    timeout_s=max(.1,float(timeout_s))
    if os.geteuid()!=0:
        return False,'sandbox unavailable: cannot drop root privileges to UID 65534'
    try:
        import ctypes
        ctypes.CDLL('libseccomp.so.2')
    except Exception as e:
        return False,'sandbox unavailable: libseccomp: '+str(e)[:200]
    with tempfile.TemporaryDirectory(prefix='e10_poetiq_r4_') as td:
        # Child cwd and TMPDIR are an isolated per-program scratch folder.
        # It is NOT a mount-based read sandbox; UID/seccomp prevent spawning,
        # outbound sockets and writing to root-owned Kaggle checkpoints.
        os.chown(td,65534,65534)
        os.chmod(td,0o700)
        output=Path(td)/'stdout.log'; errors=Path(td)/'stderr.log'
        env={'PYTHONHASHSEED':'0','PYTHONDONTWRITEBYTECODE':'1',
             'CUDA_VISIBLE_DEVICES':'','OMP_NUM_THREADS':'1',
             'OPENBLAS_NUM_THREADS':'1','MKL_NUM_THREADS':'1',
             'NUMEXPR_NUM_THREADS':'1','TMPDIR':td,
             'HOME':td, 'PATH':'/usr/bin:/bin'}
        env['E10_CHILD_CPU']=str(max(2,min(120,math.ceil(timeout_s)+1)))
        cmd=[sys.executable,'-I','-B','-c',_POETIQ_TRUSTED_RUNNER]
        try:
            with output.open('wb') as outf,errors.open('wb') as errf:
                proc=subprocess.Popen(cmd,stdin=subprocess.PIPE,stdout=outf,stderr=errf,
                      cwd=td,env=env,start_new_session=True,close_fds=True)
                try:
                    proc.communicate(input=json.dumps({'code':code,'input':grid}).encode(), timeout=timeout_s)
                except subprocess.TimeoutExpired:
                    try:os.killpg(proc.pid,signal.SIGKILL)
                    except ProcessLookupError:pass
                    proc.communicate(timeout=2)
                    return False,'sandbox timeout'
            if output.stat().st_size>1024*1024 or errors.stat().st_size>1024*1024:
                return False,'sandbox output limit exceeded'
            sout=output.read_text(encoding='utf-8',errors='replace')
            serr=errors.read_text(encoding='utf-8',errors='replace')
            if proc.returncode:
                return False,('sandbox process failure '+str(proc.returncode)+': '+(serr or sout))[-1500:]
            marker='__E10_PROGRAM_RESULT__'
            pos=sout.rfind(marker)
            if pos<0:
                return False,'sandbox missing result: '+sout[-800:]
            try: payload=json.loads(sout[pos+len(marker):].strip())
            except ValueError:
                return False,'sandbox malformed result: '+sout[-800:]
            if not payload.get('ok'):
                return False,str(payload.get('error','program error'))[:1200]
            return True,json.dumps(payload.get('result'))
        except Exception as exc:
            return False,('sandbox startup failure '+type(exc).__name__+': '+str(exc))[:1000]

def poetiq_array_diff(pred, truth):
    return '\n'.join(' '.join(str(int(a)) if a==b else f'{int(a)}/{int(b)}' for a,b in zip(pr,tr))
                     for pr,tr in zip(pred,truth))

def poetiq_verify_code(code,task,timeout_s=5.0):
    train_results=[]; per=[]
    for i,ex in enumerate(task['train']):
        ok,payload=poetiq_sandbox_run(code,ex['input'],timeout_s)
        # As in Poetiq, evaluate *partial training correctness* even when
        # predicted integer values are outside ARC 0..9; validate test grids separately.
        grid=None
        if ok:
            try:
                raw=json.loads(payload)
                # Never coerce floats, strings or bools into integer grids:
                # that can falsely mark a program train-verified and stop
                # Poetiq refinement while test candidates are invalid.
                grid=poetiq_normalize_grid(raw)
            except (TypeError,ValueError,IndexError,OverflowError):pass
        truth=ex['output']; exact=grid==truth if grid is not None else False
        if exact: soft=1.0; feedback=f'Solves Example #{i+1} correctly.'
        elif grid is None: soft=0.0;feedback=f'Solves Example #{i+1} incorrectly. Invalid or unavailable grid. Error: {payload[-600:]}'
        elif len(grid)!=len(truth) or len(grid[0])!=len(truth[0]):
            soft=0.0;feedback=f'Solves Example #{i+1} incorrectly. Shape mismatch: predicted {(len(grid),len(grid[0]))}, expected {(len(truth),len(truth[0]))}.'
        else:
            soft=sum(a==b for rowa,rowb in zip(grid,truth) for a,b in zip(rowa,rowb))/(len(truth)*len(truth[0]))
            feedback=f'Solves Example #{i+1} incorrectly. Wrong cells appear as prediction/correct:\n```\n{poetiq_array_diff(grid,truth)}\n```\nOutput accuracy: {soft:.2f}'
        per.append(soft);train_results.append({'success':bool(exact),'soft_score':soft,'feedback':feedback,'error':None if ok else payload})
    test_results=[]
    for ex in task['test']:
        ok,payload=poetiq_sandbox_run(code,ex['input'],timeout_s)
        grid=poetiq_grid_from_output(payload) if ok else None
        test_results.append({'grid':grid,'error':None if grid is not None else payload[-600:]})
    passed=all(r['success'] for r in train_results)
    test_usable=all(poetiq_valid_grid(r.get('grid')) for r in test_results)
    feedback='\n\n'.join(r['feedback'] for r in train_results)
    # Test-input truth is NEVER consulted: only runtime/format diagnostics.
    if not test_usable:
        problems=[f'Challenge #{i+1}: {r.get("error") or "invalid grid shape/palette"}'
                  for i,r in enumerate(test_results) if not poetiq_valid_grid(r.get('grid'))]
        feedback += '\n\nTEST EXECUTION / FORMAT ERRORS (no test answers are known):\n'+\
                    '\n'.join(problems)
    return {'train_results':train_results,'test_results':test_results,
            'passed':passed,'test_usable':test_usable,
            'verified_usable':bool(passed and test_usable),
            'score':float(sum(per)/len(per)) if per else 0.0,
            'feedback':feedback}

def poetiq_test_key(test_results):
    return json.dumps([r.get('grid') for r in test_results],separators=(',',':'))

def poetiq_mean_soft(res):
    trs=res.get('train_results',[])
    return sum(t.get('soft_score',0.0) for t in trs)/len(trs) if trs else 0.0

def poetiq_vote(expert_results,use_new_voting=True,count_failed_matches=True,iters_tiebreak=False,low_to_high_iters=False):
    # Poetiq's solve_parallel_coding grouping and diversity-first ranking semantics.
    passers={}; failures={}
    for result in expert_results:
        key=poetiq_test_key(result['test_results'])
        (passers if result['passed'] else failures).setdefault(key,[]).append(result)
    if use_new_voting and count_failed_matches:
        for key in list(failures):
            if key in passers:
                passers[key].extend(failures.pop(key))
    groups=list(passers.values())
    if iters_tiebreak:
        for group in groups:group.sort(key=lambda z:z.get('iteration',0), reverse=not low_to_high_iters)
        groups.sort(key=lambda g:g[0].get('iteration',0),reverse=low_to_high_iters)
    groups.sort(key=len,reverse=True)
    for group in failures.values():group.sort(key=poetiq_mean_soft,reverse=True)
    failed=list(failures.values());failed.sort(key=lambda g:(len(g),poetiq_mean_soft(g[0])),reverse=True)
    if use_new_voting:
        return ([g[0] for g in groups if g]+[g[0] for g in failed if g]+
                [m for g in groups for m in g[1:]]+[m for g in failed for m in g[1:]])
    return ([g[0] for g in groups if g]+sorted([r for g in failed for r in g],key=poetiq_mean_soft,reverse=True)
            +[m for g in groups for m in g[1:]])

def poetiq_two_attempts(task, ranked):
    # Poetiq-ranked program outputs supply both attempts; no old grid policy.
    out=[]
    for qi,test in enumerate(task['test']):
        grids=[]
        for r in ranked:
            t=r['test_results']
            g=t[qi]['grid'] if qi<len(t) else None
            if poetiq_valid_grid(g) and g not in grids:grids.append(g)
        # S1 (A2): never repeat attempt_1 (R5 did in 134/172 inputs); the next distinct grid, else the test input,
        # else its mirror
        mirror=[list(reversed(row)) for row in test['input']]
        if grids:
            a1=grids[0]
            a2=next((g for g in grids[1:]+[test['input'],mirror] if g!=a1),grids[0])
        else:
            a1=test['input'] # only if no valid program output; required schema
            a2=mirror if mirror!=a1 else test['input']
        out.append({'attempt_1':a1,'attempt_2':a2})
    return out

# --- DONOR ORIGINAL PROMPTS BELOW ---
# Source: poetiq-ai/poetiq-arc-agi-solver/arc_agi/prompts.py (MIT)
# Verbatim donor prompt texts, then loaded into Poetiq adaptation core.
SOLVER_PROMPT_1 = '''
You are an expert in solving Abstract Reasoning Corpus (ARC) tasks by writing Python code. Your goal is to analyze input-output examples and create a 'transform' function that correctly transforms any given input grid into the corresponding output grid.

Here's how to approach the problem:

**1. Analyze the Examples:**
  *   Identify the key objects in the input and output grids (e.g., shapes, lines, regions).
  *   Determine the relationships between these objects (e.g., spatial arrangement, color, size).
  *   Identify the operations that transform the input objects and relationships into the output objects and relationships (e.g., rotation, reflection, color change, object addition/removal).
  *   Consider the grid dimensions, symmetries, and other visual features.

**2. Formulate a Hypothesis:**
  *   Based on your analysis, formulate a transformation rule that works consistently across all examples.
  *   Express the rule as a sequence of image manipulation operations.
  *   Prioritize simpler rules first.
  *   Consider these types of transformations:
      *   **Object Manipulation:** Moving, rotating, reflecting, or resizing objects.
      *   **Color Changes:** Changing the color of specific objects or regions.
      *   **Spatial Arrangements:** Rearranging the objects in a specific pattern.
      *   **Object Addition/Removal:** Adding or removing objects based on certain criteria.

**3. Implement the Code:**
  *   Write a Python function called `transform(grid: np.ndarray) -> np.ndarray` that implements your transformation rule.
  *   Use NumPy for array manipulations. Other standard libraries are also available.
  *   Write modular code with clear variable names and comments to explain the logic behind each step.
  *   Document your code clearly, explaining the transformation rule in the docstring.
  *   Handle edge cases and invalid inputs gracefully.

**4. Test and Refine:**
  *   Test your code on all examples. If it fails for any example, refine your hypothesis and code.
  *   Use debugging techniques to identify and fix errors.
  *   Ensure your code handles edge cases and invalid inputs gracefully.

**5. Output:**
  *   Provide a brief explanation of your solution.
  *   Include the complete Python code for the `transform` function within a single markdown code block.
  *   Do not include any `__name__ == "__main__"` block or any code outside the function definition.

**Examples:**

**Example 1:**

**Input:**
```
[[1, 1, 1],
[1, 0, 1],
[1, 1, 1]]
```

**Output:**
```
[[0, 0, 0],
[0, 1, 0],
[0, 0, 0]]
```

**Explanation:**
Replace the border with 0s.

**Code:**
```python
import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
  """Replace the border with 0s."""
  grid[0, :] = 0
  grid[-1, :] = 0
  grid[:, 0] = 0
  grid[:, -1] = 0
  return grid
```

**Example 2:**

**Input:**
```
[[1, 2, 3],
[4, 5, 6],
[7, 8, 9]]
```

**Output:**
```
[[9, 8, 7],
[6, 5, 4],
[3, 2, 1]]
```

**Explanation:**
Reverse the order of elements in each row and then reverse the order of the rows themselves.

**Code:**
```python
import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
  """Reverses the order of elements in each row and then reverses the order of the rows."""
  new_grid = grid[:, ::-1][::-1]
  return new_grid
```

**Example 3:**

**Input:**
```
[[0, 0, 0, 0, 0],
[0, 1, 1, 1, 0],
[0, 1, 0, 1, 0],
[0, 1, 1, 1, 0],
[0, 0, 0, 0, 0]]
```

**Output:**
```
[[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 1, 0, 0],
[0, 0, 0, 0, 0],
[0, 0, 0, 0, 0]]
```

**Explanation:**
Keep only the center pixel if it is 1, otherwise make the grid all zeros.

**Code:**
```python
import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
  """Keep only the center pixel if it is 1, otherwise make the grid all zeros."""
  center_row, center_col = grid.shape[0] // 2, grid.shape[1] // 2
  if grid[center_row, center_col] == 1:
      new_grid = np.zeros_like(grid)
      new_grid[center_row, center_col] = 1
      return new_grid
  else:
      return np.zeros_like(grid)
```

**PROBLEM:**

Below is a textual representation of the input-output examples and the challenge to be solved.

$$problem$$
'''
SOLVER_PROMPT_2 = '''
You are a world-class expert in solving Abstract Reasoning Corpus (ARC) tasks. Your approach is methodical, creative, and highly effective. You are also a master Python coder, producing elegant, efficient, and well-documented solutions.

Your goal is to analyze a set of input-output examples and devise a Python function that accurately transforms any input grid into its corresponding output grid. The key is to identify a *single, consistent transformation rule* that generalizes across *all* examples. Do not give up until you find a correct solution.

Follow this iterative process:

**Part 1: Initial Analysis and Hypothesis Generation**

1.  **Example Inspection:** Carefully examine the input and output grids for each example. Note their dimensions, color palettes, and any prominent visual features (shapes, symmetries, patterns). Use visualization techniques to aid your analysis.
2.  **Transformation Hypotheses:** Formulate several candidate transformation rules. Start with simpler rules and gradually increase complexity. Consider these categories:
    *   **Color Transformations:** Replacing colors based on specific criteria (e.g., adjacency, frequency). For example, replace all 0s with 1s, or replace the most frequent color with the least frequent color.
    *   **Object Isolation:** Identifying and isolating objects based on color, shape, or position. For example, extract the largest connected component of a certain color, or isolate objects based on their spatial relationships.
    *   **Spatial Operations:** Rotating, reflecting, resizing, or moving objects. For example, rotate the grid by 90 degrees, reflect the grid horizontally or vertically, or resize the grid by a certain factor.
    *   **Pattern Generation:** Replicating or extending existing patterns. For example, repeat a certain pattern across the grid, or generate a new pattern based on the existing patterns.
3.  **Symmetry Analysis:** Identify any symmetries (rotational, reflectional) in the input and output grids. Determine if the transformation preserves or alters these symmetries.

**Part 2: Iterative Testing and Refinement**

1.  **Code Implementation:** Implement your strongest candidate rule as a Python function. The function *must* accept a 2D numpy array as input and return a 2D numpy array as output.
2.  **Rigorous Testing:** Test your code against *all* training examples. A single failure indicates an incorrect rule.
3.  **Feedback Analysis:** If your code fails, carefully analyze the feedback. Identify the specific examples that failed and the nature of the errors. Use print statements to debug intermediate values and verify your assumptions.
4.  **Hypothesis Refinement:** Based on the feedback, refine your transformation rule. This may involve adjusting parameters, adding new conditions, or discarding the rule altogether and starting with a new hypothesis.
5.  **Repeat:** Continue this iterative process of coding, testing, and refining until you find a rule that works for all training examples. Do not give up until you find a correct solution.

**Part 3: Coding Guidelines**

1.  **Available Libraries:** You can use `numpy`, `cv2` (OpenCV), and any library from the standard Python library.
2.  **Computer Vision Techniques:** Consider using `cv2` for tasks involving object detection, edge detection, or image filtering.
3.  **Utility Functions:** Write reusable utility functions to improve code modularity and readability.
4.  **Error Handling:** Implement robust error handling to gracefully manage edge cases and invalid inputs.
5.  **Code Clarity:** Write clean, well-documented code with meaningful variable names and comments.

**Part 4: Output Requirements**

1.  **Output Format:**
    *   Begin with a concise paragraph explaining the proposed solution, followed by a Python code section.
    *   You *must* provide a code output representing your best attempt. Do not give up or refuse to produce code.
    *   **The code section must be a single, valid Python code block in markdown fenced code block format and nothing else.**
    *   The main transform function must have the signature `def transform(grid: np.ndarray) -> np.ndarray`.
    *   Document the transformation rule implemented in the docstring of the transform function.
    *   Do not include any `__name__ == "__main__"` block. This will be added later by the user. You are writing a library function.

**Example:**

**Problem:**
Input:
<Diagram>
0 0 1
0 1 0
1 0 0
</Diagram>

Output:
<Diagram>
1 1 1
1 1 1
1 1 1
</Diagram>

**Explanation:**
Replace all 0s with 1s.

```python
import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    """Replace all 0s with 1s."""
    return np.where(grid == 0, 1, grid)
```

**PROBLEM:**

Below is a textual representation of the input-output examples and the challenge to be solved.

$$problem$$
'''
SOLVER_PROMPT_3 = '''
You are a world-class expert in solving Abstract Reasoning Corpus (ARC) tasks. Your approach is methodical, creative, and highly effective. You are also a master Python coder, producing elegant, efficient, and well-documented solutions.

Your goal is to analyze a set of input-output examples and devise a Python function that accurately transforms any input grid into its corresponding output grid. The key is to identify a *single, consistent transformation rule* that generalizes across *all* examples. Do not give up until you find a correct solution.

Follow this iterative process:

**Part 1: Initial Analysis and Hypothesis Generation**

1.  **Example Inspection:** Carefully examine the input and output grids for each example. Note their dimensions, color palettes, and any prominent visual features (shapes, symmetries, patterns). Use visualization techniques to aid your analysis.
2.  **Transformation Hypotheses:** Formulate several candidate transformation rules. Start with simpler rules and gradually increase complexity. Consider these categories:
    *   **Color Transformations:** Replacing colors based on specific criteria (e.g., adjacency, frequency). For example, replace all 0s with 1s, or replace the most frequent color with the least frequent color.
    *   **Object Isolation:** Identifying and isolating objects based on color, shape, or position. For example, extract the largest connected component of a certain color, or isolate objects based on their spatial relationships.
    *   **Spatial Operations:** Rotating, reflecting, resizing, or moving objects. For example, rotate the grid by 90 degrees, reflect the grid horizontally or vertically, or resize the grid by a certain factor.
    *   **Pattern Generation:** Replicating or extending existing patterns. For example, repeat a certain pattern across the grid, or generate a new pattern based on the existing patterns.
3.  **Symmetry Analysis:** Identify any symmetries (rotational, reflectional) in the input and output grids. Determine if the transformation preserves or alters these symmetries.

**Part 2: Iterative Testing and Refinement**

1.  **Code Implementation:** Implement your strongest candidate rule as a Python function. The function *must* accept a 2D numpy array as input and return a 2D numpy array as output.
2.  **Rigorous Testing:** Test your code against *all* training examples. A single failure indicates an incorrect rule.
3.  **Feedback Analysis:** If your code fails, carefully analyze the feedback. Identify the specific examples that failed and the nature of the errors. Use print statements to debug intermediate values and verify your assumptions.
4.  **Hypothesis Refinement:** Based on the feedback, refine your transformation rule. This may involve adjusting parameters, adding new conditions, or discarding the rule altogether and starting with a new hypothesis.
5.  **Repeat:** Continue this iterative process of coding, testing, and refining until you find a correct solution. Do not give up until you find a correct solution.

**Part 3: Coding Guidelines**

1.  **Available Libraries:** You can use `numpy`, `cv2` (OpenCV), and any library from the standard Python library.
2.  **Computer Vision Techniques:** Consider using `cv2` for tasks involving object detection, edge detection, or image filtering.
3.  **Utility Functions:** Write reusable utility functions to improve code modularity and readability.
4.  **Error Handling:** Implement robust error handling to gracefully manage edge cases and invalid inputs.
5.  **Code Clarity:** Write clean, well-documented code with meaningful variable names and comments. The code should be as concise as possible.

**Part 4: Output Requirements**

1.  **Output Format:**
    *   Begin with a concise paragraph explaining the proposed solution, followed by a Python code section.
    *   You *must* provide a code output representing your best attempt. Do not give up or refuse to produce code.
    *   **The code section must be a single, valid Python code block in markdown fenced code block format and nothing else.**
    *   The main transform function must have the signature `def transform(grid: np.ndarray) -> np.ndarray`.
    *   Document the transformation rule implemented in the docstring of the transform function.
    *   Do not include any `__name__ == "__main__"` block. This will be added later by the user. You are writing a library function.

**Example 1:**

**Problem:**
Input:
<Diagram>
0 0 1
0 1 0
1 0 0
</Diagram>

Output:
<Diagram>
1 1 1
1 1 1
1 1 1
</Diagram>

**Explanation:**
Replace all 0s with 1s.

```python
import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    """Replace all 0s with 1s."""
    return np.where(grid == 0, 1, grid)
```

**Example 2:**

**Problem:**
Input:
<Diagram>
0 0 0
0 1 0
0 0 0
</Diagram>

Output:
<Diagram>
0 1 0
1 1 1
0 1 0
</Diagram>

**Explanation:**
Replace all neighbors of 1 with 1.

```python
import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    """Replace all neighbors of 1 with 1."""
    new_grid = grid.copy()
    for i in range(1, grid.shape[0] - 1):
        for j in range(1, grid.shape[1] - 1):
            if grid[i][j] == 1:
                new_grid[i-1][j] = 1
                new_grid[i+1][j] = 1
                new_grid[i][j-1] = 1
                new_grid[i][j+1] = 1
    return new_grid
```

**Example 3:**

**Problem:**
Input:
<Diagram>
1 2 3
4 5 6
7 8 9
</Diagram>

Output:
<Diagram>
9 8 7
6 5 4
3 2 1
</Diagram>

**Explanation:**
Reverse the grid.

```python
import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    """Reverse the grid."""
    return np.flip(grid)
```

**PROBLEM:**

Below is a textual representation of the input-output examples and the challenge to be solved.

$$problem$$
'''
FEEDBACK_PROMPT = '''
**EXISTING PARTIAL/INCORRECT SOLUTIONS:**

Following are some of the best, though not completely correct, solutions so far. For each solution, its code, corresponding feedback regarding its output on the example problems, and a numeric score between 0. (worst) and 1. (best) indicating the quality of outputs is also provided. Study these solutions and corresponding feedback and produce a new solution fixing all the issues. Make sure to follow the output format specified earlier.

$$feedback$$
'''

# Replace only the textual prompt templates with the published donor originals.
POETIQ_SOLVER_PROMPTS = {1:SOLVER_PROMPT_1, 2:SOLVER_PROMPT_2, 3:SOLVER_PROMPT_3}
POETIQ_FEEDBACK_PROMPT = FEEDBACK_PROMPT

# A positive runtime test executes before vLLM engine loading (notebook Cell 6).
# Never claim readiness based on a seccomp library import alone.
def poetiq_sandbox_selftest():
    import tempfile
    test_grid=[[2]]
    ok, payload=poetiq_sandbox_run(
        'import numpy as np\ndef transform(grid):\n return np.asarray(grid)', test_grid, 8)
    if not ok or poetiq_grid_from_output(payload)!=test_grid:
        raise RuntimeError('Poetiq sandbox positive test FAILED: '+str(payload)[:500])
    with tempfile.TemporaryDirectory(prefix='e10_root_protected_') as td:
        protected=Path(td)/'no_write'
        protected.write_text('original',encoding='utf-8')
        protected.chmod(0o600)
        attack=f"def transform(grid):\n open({str(protected)!r}, 'w').write('MODIFIED')\n return grid"
        ok,err=poetiq_sandbox_run(attack,test_grid,8)
        if ok or protected.read_text(encoding='utf-8')!='original':
            raise RuntimeError('Poetiq sandbox protected-file write test FAILED')
    for label,attack in (
        ('spawn',"import subprocess\ndef transform(grid):\n subprocess.check_output(['/usr/bin/id'])\n return grid"),
        ('network',"import socket\ndef transform(grid):\n socket.socket()\n return grid"),
    ):
        ok,err=poetiq_sandbox_run(attack,test_grid,8)
        if ok or 'Operation not permitted' not in str(err):
            raise RuntimeError('Poetiq sandbox '+label+' restriction FAILED: '+str(err)[:300])
    return {'ok':True,'mode':'uid_drop+seccomp+rlimits+timeout',
            'filesystem_isolation':'NOT a private mount; world-readable host paths may be readable',
            'verified':['numpy_transform','protected_file_write_blocked','spawn_blocked','network_blocked']}

# Fail closed without losing the cell-0 competition-format fallback.
try:
    E10_SANDBOX_SELFTEST=poetiq_sandbox_selftest()
except BaseException as _e10_sandbox_error:
    E10_SANDBOX_SELFTEST={'ok':False,'mode':'unavailable',
                         'error':type(_e10_sandbox_error).__name__+': '+str(_e10_sandbox_error)[:1200]}
print('[E10] Sandbox self-test:',E10_SANDBOX_SELFTEST,flush=True)


In [ ]:
# ==================================================================== CELL E19-J - verdict rules (pure functions, no GPU, no globals)
# E19: does vLLM 0.19's MTP + prefix caching (and fp8 KV) silently corrupt Qwen3.8-27B greedy outputs on the S1 stack?
# This cell holds ONLY pure functions over saved rows, so the same code judges the rows inside the notebook and offline
# (src/e19_compare.py imports this file). Rows are dicts saved one per line in q27x/e19/e19_rows.jsonl.
# Rules:
#   - rows are compared by integer token id; the first divergence is a LARGE-MARGIN FLIP when a known top-1 minus top-2
#     logprob margin at that position (either run) is > 1.0 nat (the Experiment 14 VMS / EMS gate); a divergence with no
#     margin > 1 and a missing margin is missing evidence (never a pass); low-margin divergences are tolerated (greedy +
#     MTP is not bit-exact across batch compositions);
#   - a corrupt row = empty output, degenerate text (32+ identical characters, < 8 distinct characters over 200, < 10 %
#     letters / digits; ARC grids excluded), a run of 64+ identical token ids, 8+ CJK characters, or a greedy token that
#     is not the top-1 of its own logprobs;
#   - the main discriminator is WITHIN the arm: serial cold vs cold repeat (the arm's own noise), serial cold vs warm
#     prefix hit, serial vs churn-concurrent (batch reordering), natural vs continuation over cached generated tokens,
#     plus MTP acceptance per /metrics window (collapse to ~0);
#   - the churn check is calibrated by the noise of arms A (MTP off, prefix off) and B (MTP on, prefix off); the token
#     comparison against arm A is informational only;
#   - an arm with planned rows not run because of the time budget is NOT_TESTED_BUDGET, never PASS.
import json as _json
import math as _math
import re as _re

E19_FLIP_NAT = 1.0
E19_MARGIN_COVERAGE = 0.95        # rev 2 (audit B4): share of a row's tokens that must carry a finite top-1 - top-2 margin
E19_SAME_BUDGET = ("aa_serial", "cache_serial")   # + every cont_<stage>: same max_tokens on both sides (rev 2, audit B2)
E19_BUDGET_STATUSES = ("not_tested_budget", "missing")
E19_STATUS_ORDER = ["FAIL", "LAUNCH_FAILED", "INCOMPLETE", "NOT_TESTED_BUDGET", "INCONCLUSIVE", "PASS"]
_CJK_RANGES = ((0x3040, 0x30FF), (0x3400, 0x4DBF), (0x4E00, 0x9FFF), (0xAC00, 0xD7AF), (0xF900, 0xFAFF))
_CJK = _re.compile("[" + "".join(chr(a) + "-" + chr(b) for a, b in _CJK_RANGES) + "]")


def e19_grid_like(s):
    """VMS rev 2g: an ARC grid by structure (digits / brackets / commas / whitespace, every row <= 30 cells)."""
    s = (s or "").strip()
    return bool(_re.fullmatch(r"[0-9,\[\]\s]+", s)) and len(_re.findall(r"\d", s)) >= 2 \
        and all(len(_re.findall(r"\d", r_)) <= 30 for r_ in _re.split(r"\]|\n", s))


def e19_text_bad(x):
    """VMS rev 2e-c: empty, or (not grid-like) a 32-character run of one character, < 8 distinct characters over 200,
    or < 10 % letters / digits."""
    s = (x or "").strip()
    if not s:
        return True
    if e19_grid_like(s):
        return False
    if _re.search(r"(.)\1{31,}", s):
        return True
    return len(s) >= 200 and (len(set(s)) < 8 or sum(ch.isalnum() for ch in s) / len(s) < 0.10)


def e19_token_run(ids, n=64):
    run, prev = 0, None
    for t in ids or []:
        run = run + 1 if t == prev else 1
        prev = t
        if run >= n:
            return True
    return False


def e19_row_flags(r):
    """Corruption flags of one completed row (text = the decoded output, saved by the client)."""
    toks, text = list(r.get("tokens") or []), r.get("text") or ""
    f = {"empty": not toks or not text.strip(),
         "degenerate": bool(toks) and e19_text_bad(text),
         "token_run": e19_token_run(toks),
         "cjk": len(_CJK.findall(text)) >= 8,
         "argmax_mismatch": len(r.get("top1_mismatch") or [])}
    f["corrupt"] = bool(f["empty"] or f["degenerate"] or f["token_run"] or f["cjk"] or f["argmax_mismatch"])
    return f


def e19_margin_coverage(r):
    """rev 2 (audit B4): share of the row's tokens with a finite top-1 - top-2 logprob margin."""
    toks, mg = list(r.get("tokens") or []), list(r.get("margins") or [])
    if not toks or len(mg) != len(toks):
        return 0.0
    return sum(1 for x in mg if isinstance(x, (int, float)) and not isinstance(x, bool) and _math.isfinite(x)) / len(toks)


def e19_key(r):
    return (r.get("stage"), r.get("pass"), r.get("idx"))


def e19_ok(r):
    return r is not None and r.get("status") == "ok"


def e19_index(rows, arm=None):
    """Last row per key (a row is written once; the index is defensive against duplicates)."""
    out = {}
    for r in rows:
        if arm is None or r.get("arm") == arm:
            out[e19_key(r)] = r
    return out


def e19_first_div(ta, ma, tb, mb):
    """Compare two token lists (+ margins aligned to them). Returns one comparison record."""
    ta, tb = list(ta or []), list(tb or [])
    n = min(len(ta), len(tb))
    k = next((j for j in range(n) if ta[j] != tb[j]), None)
    if k is None:
        # identical over the common length; a length difference alone (one stopped earlier) is not a token flip
        return {"identical": len(ta) == len(tb), "first_divergence": None if len(ta) == len(tb) else n,
                "length_only": len(ta) != len(tb), "large_margin_flip": False, "missing_margin": False,
                "margin_a": None, "margin_b": None, "compared_len": n}
    ma_k = (ma or [None] * (k + 1))[k] if k < len(ma or []) else None
    mb_k = (mb or [None] * (k + 1))[k] if k < len(mb or []) else None
    known = [m for m in (ma_k, mb_k) if isinstance(m, (int, float))]
    big = any(m > E19_FLIP_NAT for m in known)
    return {"identical": False, "first_divergence": k, "length_only": False, "large_margin_flip": big,
            "missing_margin": (not big) and len(known) < 2, "margin_a": ma_k, "margin_b": mb_k, "compared_len": n}


def e19_pair_view(r, part):
    """Token / margin view of a row for a comparison. part 'all' = the whole output; 'tail' = the natural row after its
    first g_pre tokens (what the continuation request must reproduce)."""
    toks, mg = list(r.get("tokens") or []), list(r.get("margins") or [])
    if part == "tail":
        g = int(r.get("g_pre") or 0)
        return toks[g:], mg[g:]
    return toks, mg


def e19_compare(pairs):
    """pairs: list of (row_a, part_a, row_b, part_b). Missing / failed rows are counted as not compared."""
    per, not_compared, nc_budget = [], 0, 0
    for ra, pa, rb, pb in pairs:
        if not (e19_ok(ra) and e19_ok(rb)):
            not_compared += 1
            if any(r_ is None or r_.get("status") in E19_BUDGET_STATUSES for r_ in (ra, rb)):
                nc_budget += 1
            continue
        ta, ma = e19_pair_view(ra, pa)
        tb, mb = e19_pair_view(rb, pb)
        if not ta or not tb:
            not_compared += 1
            continue
        c = e19_first_div(ta, ma, tb, mb)
        c["idx"] = ra.get("idx")
        per.append(c)
    div = [c for c in per if not c["identical"] and not c["length_only"]]
    fd = [c["first_divergence"] for c in div]
    return {"planned": len(pairs), "compared": len(per), "not_compared": not_compared,
            "not_compared_budget": nc_budget, "not_compared_other": not_compared - nc_budget,
            "identical": sum(1 for c in per if c["identical"]), "length_only": sum(1 for c in per if c["length_only"]),
            "differ": len(div), "large_margin_flips": sum(1 for c in div if c["large_margin_flip"]),
            "missing_margins": sum(1 for c in div if c["missing_margin"]),
            "first_divergence": fd, "flip_margins": [[c["margin_a"], c["margin_b"]] for c in div],
            "per": per}


def e19_churn_ref(ix, pc, r):
    """The serial reference of a churn row: the same prompt alone in the same arm, from the cache when the churn row was
    served from the cache (prefix-ON arm), else cold."""
    if pc and (r.get("cached_tokens") or 0) > 0:
        return ix.get(("serial", "warm", r.get("idx")))
    return ix.get(("serial", "cold", r.get("idx")))


def e19_arm_pairs(ix, meta):
    """The within-arm comparisons (the main discriminator).
    aa_serial    - serial cold vs serial cold repeat (prefix-ON: cache reset before each; prefix-OFF: the warm pass is a
                   second cold run, nothing is cached): the arm's own numeric noise;
    cache_serial - serial cold vs serial warm (prefix-cache hit), prefix-ON arms only;
    churn        - every churn row (12-16 concurrent requests, staggered lengths and arrivals: the batch reorders
                   continuously) vs the same prompt run alone (serial) in the same arm, over their common length;
    cont_<stage> - the natural continuation's tail vs the continuation request over the server's own cached tokens."""
    pc = bool(meta.get("prefix_caching"))
    ser = list(meta.get("serial_idx") or [])
    g = lambda st, ps, i: ix.get((st, ps, i))
    out = {"aa_serial": [(g("serial", "cold", i), "all", g("serial", "cold2" if pc else "warm", i), "all") for i in ser]}
    if pc:
        out["cache_serial"] = [(g("serial", "cold", i), "all", g("serial", "warm", i), "all") for i in ser]
    out["churn"] = []
    for idx, rep, _m in meta.get("churn_jobs") or []:
        r = g("churn", f"j{rep}", idx)
        out["churn"].append((e19_churn_ref(ix, pc, r) if r else None, "all", r, "all"))
    for st in meta.get("long_stages") or []:
        out["cont_" + st["name"]] = [(g(st["name"], "nat", i), "tail", g(st["name"], "cont", i), "all") for i in st["idx"]]
    return out


def e19_planned_keys(meta):
    pc = bool(meta.get("prefix_caching"))
    keys = []
    for i in meta.get("serial_idx") or []:
        keys += [("serial", "cold", i), ("serial", "warm", i)] + ([("serial", "cold2", i)] if pc else [])
    keys += [("churn", f"j{rep}", idx) for idx, rep, _m in meta.get("churn_jobs") or []]
    for st in meta.get("long_stages") or []:
        for i in st["idx"]:
            keys += [(st["name"], "nat", i), (st["name"], "cont", i)]
    return keys


def e19_worst(*statuses):
    s = [x for x in statuses if x]
    return min(s, key=E19_STATUS_ORDER.index) if s else "PASS"


def e19_accept(meta):
    """MTP acceptance per /metrics window (accepted / draft tokens). Collapse = a window with >= 300 draft tokens and an
    acceptance below 0.10 (Qwen3.8 MTP 3 normally accepts about half of the draft tokens)."""
    w = [x for x in (meta.get("accept_windows") or []) if (x.get("draft_tokens") or 0) >= 300]
    rates = [round((x.get("accepted") or 0) / x["draft_tokens"], 3) for x in w]
    return {"windows": len(w), "min": min(rates) if rates else None,
            "median": sorted(rates)[len(rates) // 2] if rates else None, "collapsed": sum(1 for r_ in rates if r_ < 0.10)}


def e19_judge_arm(rows, meta):
    """In-arm verdict of one arm from its rows and meta. The churn comparison is NOT judged here (e19_verdicts() judges
    it against the noise of the calibration arms A / B, or strictly against the arm's own serial A/A)."""
    if meta.get("launch_failed"):
        return {"arm": meta.get("arm"), "verdict": "LAUNCH_FAILED", "reasons": [meta.get("launch_failed")], "comparisons": {},
                "corrupt_rows": [], "aa_lmf": None, "churn_lmf": None, "churn_compared": 0}
    ix = e19_index(rows, meta.get("arm"))
    planned = e19_planned_keys(meta)
    pc = bool(meta.get("prefix_caching"))
    st_count = {}
    for k in planned:
        s_ = (ix.get(k) or {}).get("status", "missing")
        st_count[s_] = st_count.get(s_, 0) + 1
    reasons = []
    flags = {k: e19_row_flags(ix[k]) for k in planned if e19_ok(ix.get(k))}
    corrupt_rows = [list(k) for k, f in flags.items() if f["corrupt"]]
    partner = {}
    for k in planned:
        st, ps, i = k
        if ps in ("warm", "cold2"):
            partner[k] = (st, "cold", i)
        elif ps == "cont":
            partner[k] = (st, "nat", i)
        elif st == "churn" and e19_ok(ix.get(k)):
            ref = e19_churn_ref(ix, pc, ix[k])
            partner[k] = e19_key(ref) if ref else None
    hit_corrupt = [list(k) for k, p in partner.items()
                   if p and k in flags and p in flags and flags[k]["corrupt"] and not flags[p]["corrupt"]]
    cached = lambda k: (ix.get(k) or {}).get("cached_tokens")
    warm_keys = [k for k in planned if k[1] in ("warm", "cont") and e19_ok(ix.get(k))]
    cold_keys = [k for k in planned if k[0] == "serial" and k[1] in ("cold", "cold2") and e19_ok(ix.get(k))]
    churn_keys = [k for k in planned if k[0] == "churn" and e19_ok(ix.get(k))]
    cache_ev = {"warm_rows": len(warm_keys), "warm_hits": sum(1 for k in warm_keys if (cached(k) or 0) > 0),
                "cold_rows": len(cold_keys), "cold_hits": sum(1 for k in cold_keys if (cached(k) or 0) > 0),
                "churn_rows": len(churn_keys), "churn_hits": sum(1 for k in churn_keys if (cached(k) or 0) > 0),
                "cont_generated_in_hit": [((cached(k) or 0) - int((ix.get(k) or {}).get("pf_len") or 0))
                                          for k in planned if k[1] == "cont" and e19_ok(ix.get(k))],
                "usage_seen": sum(1 for k in planned if (ix.get(k) or {}).get("usage_seen"))}
    comps = {name: e19_compare(pairs) for name, pairs in e19_arm_pairs(ix, meta).items()}
    aa_lmf = comps["aa_serial"]["large_margin_flips"]
    acc = e19_accept(meta)
    status = "PASS"
    if hit_corrupt:
        status = "FAIL"
        reasons.append(f"corrupt rows (cache hit / churn / continuation) whose clean partner is not corrupt: {hit_corrupt[:8]}")
    for nm in comps:
        if nm in ("aa_serial", "churn"):
            continue
        if comps[nm]["large_margin_flips"] > aa_lmf:
            status = "FAIL"
            reasons.append(f"{nm}: {comps[nm]['large_margin_flips']} large-margin flips > the arm's serial A/A {aa_lmf}")
    if meta.get("spec") and acc["collapsed"]:
        status = "FAIL"
        reasons.append(f"MTP acceptance collapsed in {acc['collapsed']} of {acc['windows']} /metrics windows (min {acc['min']})")
    # rev 2 (audit B1): corrupt outputs without a clean partner are never a clean PASS
    if status != "FAIL" and corrupt_rows:
        status = e19_worst(status, "INCONCLUSIVE")
        reasons.append(f"{len(corrupt_rows)} anomalous outputs with no clean partner to establish relative corruption: "
                       f"{corrupt_rows[:8]}")
    if status != "FAIL":
        # rev 2 (audit B2): an unexplained length mismatch between same-budget rows (one stopped early) is not a PASS
        for nm, c in comps.items():
            if (nm in E19_SAME_BUDGET or nm.startswith("cont_")) and c.get("length_only", 0) > 0:
                status = e19_worst(status, "INCONCLUSIVE")
                reasons.append(f"{nm}: {c['length_only']} unexplained output-length mismatches (same max_tokens)")
        # rev 2 (audit B3): rows with an unvalidated status (e.g. na_short) and pairs not compared for a non-budget reason
        unexpected = {k_: v for k_, v in st_count.items() if k_ not in ("ok", "error", "server_dead") + E19_BUDGET_STATUSES}
        if unexpected:
            status = e19_worst(status, "INCOMPLETE")
            reasons.append(f"unvalidated row statuses: {unexpected}")
        nco = {nm: c["not_compared_other"] for nm, c in comps.items() if c.get("not_compared_other")}
        if nco:
            status = e19_worst(status, "INCOMPLETE")
            reasons.append(f"pairs not compared (non-budget): {nco}")
        # rev 2 (audit B4): every completed row needs logprob-margin evidence
        thin = [list(k) for k in planned if e19_ok(ix.get(k)) and e19_margin_coverage(ix[k]) < E19_MARGIN_COVERAGE]
        if thin:
            status = e19_worst(status, "INCOMPLETE")
            reasons.append(f"insufficient logprob-margin evidence (< {E19_MARGIN_COVERAGE:.0%} of tokens) in {len(thin)} rows")
        # rev 2 (audit 8): the cold controls need a cache reset that answered 200 and usage records that report cache use
        if pc:
            bad_reset = [list(k) for k in planned if k[0] == "serial" and k[1] in ("cold", "cold2") and e19_ok(ix.get(k))
                         and (ix[k].get("reset") not in (200, None) or ix[k].get("reset") is None)]
            if bad_reset:
                status = e19_worst(status, "INCONCLUSIVE")
                reasons.append(f"cache reset not confirmed (HTTP 200) before {len(bad_reset)} cold rows")
        if st_count.get("ok") and not cache_ev["usage_seen"]:
            status = e19_worst(status, "INCONCLUSIVE")
            reasons.append("no row carried a usage record (cached tokens unknown): cache evidence missing")
    if status != "FAIL":
        if meta.get("server_died"):
            status = e19_worst(status, "INCOMPLETE")
            reasons.append("server died during the arm: " + str(meta.get("server_died"))[:300])
        n_err = sum(v for k_, v in st_count.items() if k_ in ("error", "server_dead"))
        if n_err:
            status = e19_worst(status, "INCOMPLETE")
            reasons.append(f"{n_err} planned rows errored / lost to the server")
        if any(c["missing_margins"] for c in comps.values()):
            status = e19_worst(status, "INCOMPLETE")
            reasons.append("divergences without a usable logprob margin (missing evidence)")
        n_bud = st_count.get("not_tested_budget", 0) + st_count.get("missing", 0)
        if n_bud:
            status = e19_worst(status, "NOT_TESTED_BUDGET")
            reasons.append(f"{n_bud} of {len(planned)} planned rows not run (time budget)")
        if meta.get("spec") and not acc["windows"] and st_count.get("ok"):
            status = e19_worst(status, "INCONCLUSIVE")
            reasons.append("MTP arm without a single /metrics acceptance window (spec-decode counters missing)")
        if pc and cache_ev["warm_rows"] and cache_ev["warm_hits"] == 0:
            status = e19_worst(status, "INCONCLUSIVE")
            reasons.append("prefix caching ON but no warm / continuation row reported cached tokens: the cache was not exercised")
        if pc and cache_ev["cold_hits"]:
            status = e19_worst(status, "INCONCLUSIVE")
            reasons.append(f"{cache_ev['cold_hits']} serial cold rows were served from the cache (reset not effective)")
        if not pc and cache_ev["warm_hits"] + cache_ev["cold_hits"] + cache_ev["churn_hits"]:
            status = e19_worst(status, "INCONCLUSIVE")
            reasons.append("prefix caching OFF but rows reported cached tokens")
    gen_reuse = any(x > 0 for x in cache_ev["cont_generated_in_hit"])
    cache_ev["trailing_mamba_cache_tested"] = gen_reuse      # rev 2 (audit 7): NOT_EXERCISED when False
    if pc and not gen_reuse:
        reasons.append("trailing Mamba block over generated tokens NOT_EXERCISED (no continuation hit reached into the "
                       "server's generated tokens): no safety claim for that path")
    return {"arm": meta.get("arm"), "verdict": status, "reasons": reasons, "planned_rows": len(planned),
            "row_status": st_count, "corrupt_rows": corrupt_rows, "cache_hit_corrupt": hit_corrupt, "cache": cache_ev,
            "accept": acc, "aa_lmf": aa_lmf, "churn_lmf": comps["churn"]["large_margin_flips"],
            "churn_compared": comps["churn"]["compared"],
            "comparisons": {k: {kk: vv for kk, vv in v.items() if kk != "per"} for k, v in comps.items()}}


def e19_vs_ref(rows_x, meta_x, rows_ref, meta_ref):
    """SECONDARY, informational: arm X vs arm A token by token on rows with identical prompt ids. MTP on vs off can
    legitimately differ at low-margin points; this never decides a verdict."""
    ix, ir = e19_index(rows_x, meta_x.get("arm")), e19_index(rows_ref, meta_ref.get("arm"))
    groups, mismatch = {}, 0
    for k, r in ix.items():
        rr = ir.get(k)
        if rr is None or not (e19_ok(r) and e19_ok(rr)):
            continue
        if r.get("prompt_sha") != rr.get("prompt_sha"):
            mismatch += 1
            continue
        groups.setdefault(k[0] + "/" + ("churn" if k[0] == "churn" else k[1]), []).append((rr, "all", r, "all"))
    comps = {g: e19_compare(p) for g, p in sorted(groups.items())}
    return {"prompt_mismatch_rows": mismatch,
            "comparisons": {k: {kk: vv for kk, vv in v.items() if kk not in ("per", "flip_margins")} for k, v in comps.items()}}


E19_CALIBRATORS = ("A", "B")      # A = MTP off + prefix off (pure batch noise); B = MTP on + prefix off (MTP batch noise)


def e19_verdicts(rows, metas):
    """Every arm: the in-arm verdict, then the churn and corrupt-row checks calibrated by the noise of arms A / B.
    Calibration: A qualifies when its in-arm verdict is not FAIL / LAUNCH_FAILED, it compared churn rows and its churn
    flips <= its own serial A/A flips; B qualifies likewise with its churn flips <= A's (or, without A, <= its own A/A).
    An arm's churn ceiling = the max churn large-margin flips over the OTHER qualified calibrators (corrupt rows
    likewise); with none, the arm is judged strictly against its own serial A/A ('uncalibrated').
    Final = the worse of the in-arm and the churn verdicts. The token comparison against A is informational."""
    J = {n: e19_judge_arm(rows, m) for n, m in metas.items()}
    good = lambda n: n in J and J[n]["verdict"] == "PASS" and (J[n].get("churn_compared") or 0) > 0   # rev 2
    qual = {}
    if good("A") and J["A"]["churn_lmf"] <= J["A"]["aa_lmf"]:
        qual["A"] = J["A"]
    if good("B") and J["B"]["churn_lmf"] <= (J["A"]["churn_lmf"] if "A" in qual else J["B"]["aa_lmf"]):
        qual["B"] = J["B"]
    out = {}
    for n, j in J.items():
        ch = {"calibrators": sorted(q for q in qual if q != n), "reasons": []}
        if j["verdict"] == "LAUNCH_FAILED" or not j.get("churn_compared"):
            ch["verdict"] = None          # nothing compared: the in-arm status (launch / budget / incomplete) speaks
        else:
            others = [q for q in qual if q != n]
            cf = max((qual[q]["churn_lmf"] for q in others), default=None)
            cc = max((len(qual[q]["corrupt_rows"]) for q in others), default=None)
            lim = cf if cf is not None else j["aa_lmf"]
            ch.update(mode="calibrated" if cf is not None else "uncalibrated", ceiling_flips=lim, ceiling_corrupt=cc,
                      verdict="PASS")
            if j["churn_lmf"] > lim:
                ch["verdict"] = "FAIL"
                ch["reasons"].append(f"churn: {j['churn_lmf']} large-margin flips vs serial > {ch['mode']} ceiling {lim}")
            if cc is not None and len(j["corrupt_rows"]) > cc:
                ch["verdict"] = "FAIL"
                ch["reasons"].append(f"{len(j['corrupt_rows'])} corrupt rows > the calibrators' {cc}")
        vs = (e19_vs_ref(rows, metas[n], rows, metas["A"]) if "A" in metas and n != "A"
              and J["A"]["verdict"] != "LAUNCH_FAILED" else None)
        out[n] = {"final": e19_worst(j["verdict"], ch["verdict"]), "in_arm": j, "churn": ch, "vs_A_info": vs}
    return out


def e19_fmt(c, k):
    return f"{c[k]['identical']}/{c[k]['compared']} ({c[k]['large_margin_flips']})" if k in c else "-"


def e19_report_lines(V, arms_cfg, state_arms, title):
    L = [title, "",
         "| arm | what | final | in-arm | churn (mode) | rows ok / planned | serial A/A ident/cmp (flips) | "
         "cache serial | churn vs serial | cont8k | cont30k | corrupt rows | MTP accept min / median | cache warm hits | "
         "launch s | KV tokens | block |", "|" + "---|" * 17]
    for name, v in V.items():
        j, ch, cfg = v["in_arm"], v["churn"], arms_cfg.get(name, {})
        c, a = j.get("comparisons") or {}, state_arms.get(name, {})
        acc, cache = j.get("accept") or {}, j.get("cache") or {}
        L.append(f"| {name} | {cfg.get('what', '')} | **{v['final']}** | {j['verdict']} | {ch.get('verdict') or '-'} "
                 f"({ch.get('mode', '-')}) | {(j.get('row_status') or {}).get('ok', 0)}/{j.get('planned_rows', '-')} | "
                 f"{e19_fmt(c, 'aa_serial')} | {e19_fmt(c, 'cache_serial')} | {e19_fmt(c, 'churn')} | "
                 f"{e19_fmt(c, 'cont_long8k')} | {e19_fmt(c, 'cont_long30k')} | {len(j.get('corrupt_rows') or [])} | "
                 f"{acc.get('min', '-')} / {acc.get('median', '-')} | {cache.get('warm_hits', '-')}/{cache.get('warm_rows', '-')} | "
                 f"{(a.get('launch') or {}).get('sec', '-')} | {(a.get('log') or {}).get('kv_tokens', '-')} | "
                 f"{(a.get('log') or {}).get('block_size', '-')} |")
    L += ["", "## Reasons", ""]
    for name, v in V.items():
        rs = (v["in_arm"].get("reasons") or []) + (v["churn"].get("reasons") or [])
        L.append(f"- **{name}** ({v['final']}): " + ("; ".join(rs) or "none"))
    L += ["", "## Secondary: token comparison against arm A (informational; MTP on vs off may differ at low margins)", ""]
    for name, v in V.items():
        if v.get("vs_A_info"):
            L.append(f"- {name}: " + "; ".join(f"{g} {e19_fmt(v['vs_A_info']['comparisons'], g)}"
                                              for g in sorted(v["vs_A_info"]["comparisons"])))
    L += ["", "Cells: identical / compared (large-margin flips). A large-margin flip = the first token divergence where a "
          "known top-1 minus top-2 logprob margin is > 1 nat. In-arm FAIL: corrupt rows with clean partners, cache-hit or "
          "continuation flips above the arm's serial A/A, MTP acceptance collapse. Churn FAIL: flips vs serial above the "
          "calibrators' (A / B) churn flips, or above the arm's own serial A/A when no calibrator ran ('uncalibrated'). "
          "Rows not run in the budget make an arm NOT_TESTED_BUDGET, never PASS."]
    return L


def e19_load_rows(path):
    rows = []
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if line:
                try:
                    rows.append(_json.loads(line))
                except ValueError:
                    pass          # a torn last line after a crash
    return rows


In [ ]:
# ==================================================================== CELL E19-1 — vLLM 0.19 OpenAI server per arm, streaming greedy client
# E19 (2026-10-27): the S1-60K engine (vLLM 0.19.0, FP8 repack, TP4, fp8_e4m3 KV, MTP 3, FLASHINFER if -lcuda links,
# prefix caching, 4,096 batched tokens, mem 0.90, custom all-reduce off, language-model-only, max_model_len 106,496,
# seed 20261005, 16 seqs) is launched as an OpenAI server from the kernel's own Python (the vLLM 0.19 that CELL 2
# installed), one server per arm, killed between arms. Only the arm's own switch changes (MTP on/off, prefix caching
# on/off, KV dtype). Two frontend-only additions on EVERY arm: --enable-prompt-tokens-details (cached tokens in the
# usage chunk) and VLLM_SERVER_DEV_MODE=1 (POST /reset_prefix_cache); neither changes the engine's computation.
# Arm E adds --no-async-scheduling (suspect: the async-scheduling race in the V1 model runner, vLLM #53919).
# Requests: streaming /v1/completions with the prompt token ids, temperature 0 (greedy), logprobs 2 with
# return_tokens_as_token_ids (top-1 / top-2 margins keyed by token id), return_token_ids (integer ids per chunk) and
# stream_options.include_usage (prompt_tokens_details.cached_tokens).
import shlex, socket, signal as _signal, urllib.request as _urlreq, statistics
import shutil as _shutil

E19_FAKE = os.environ.get("E19_FAKE_SERVER")          # local CPU harness only: a fake vLLM 0.19 OpenAI server script
E19_PORT = int(os.environ.get("E19_PORT", 8019))
E19_URL = f"http://127.0.0.1:{E19_PORT}"
E19_DIR = OUT / "e19"
E19_DIR.mkdir(parents=True, exist_ok=True)
E19_ROWS_PATH = E19_DIR / "e19_rows.jsonl"
E19_STATE_PATH = E19_DIR / "e19_state.json"
E19_SILENCE_SEC = float(os.environ.get("E19_SILENCE_SEC", 600))
E19_READY_MAX_SEC = float(os.environ.get("E19_READY_MAX_SEC", 25 * 60))
E19_LOCK = threading.Lock()
E19_SAVE_LOCK = threading.Lock()
E19_STATE = {"run_id": RUN_ID, "profile": PROFILE, "arms": {}, "order": [], "events": []}
E19_SERVER = None

# S1 CELL 5, verbatim:
def ensure_libcuda_linkable(candidates=None, workdir="/tmp/q27x_libcuda_shim"):
    info = {"needed": None, "real": None, "shim": None, "probe": "FAIL"}
    try:
        work = Path(workdir); work.mkdir(parents=True, exist_ok=True)
        if candidates is None:
            existing = [Path(d, "libcuda.so") for d in ("/usr/local/cuda/lib64", "/usr/local/cuda/lib64/stubs",
                                                        "/usr/lib/x86_64-linux-gnu", "/usr/lib64")]
            existing = [p for p in existing if p.exists()]
        else:
            existing = []
        if existing:
            info.update(needed=False, real=str(existing[0]))
        else:
            cands = list(candidates) if candidates is not None else []
            if candidates is None:
                for d in ("/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64",
                          "/usr/local/cuda/compat"):
                    try:
                        cands += sorted(str(p) for p in Path(d).glob("libcuda.so.*") if p.is_file())
                    except Exception:
                        pass
                try:
                    for line in subprocess.run(["ldconfig", "-p"], capture_output=True, text=True, timeout=30).stdout.splitlines():
                        if "libcuda.so.1" in line and "=>" in line:
                            cands.append(line.split("=>")[-1].strip())
                except Exception:
                    pass
            real = next((Path(c) for c in cands if Path(c).is_file()), None)
            if real is None:
                info["reason"] = "libcuda.so.1 not found"
                return info
            link = work / "libcuda.so"
            if link.exists() or link.is_symlink():
                link.unlink()
            try:
                link.symlink_to(real.resolve())
            except OSError:
                _shutil.copyfile(real, link)
            old = os.environ.get("LIBRARY_PATH", "")
            os.environ["LIBRARY_PATH"] = f"{work}:{old}" if old else str(work)
            info.update(needed=True, real=str(real), shim=str(link), library_path=os.environ["LIBRARY_PATH"])
        cxx = _shutil.which("c++") or _shutil.which("g++")
        if not cxx:
            info["reason"] = "no C++ compiler for the link probe"
            return info
        cp = subprocess.run([cxx, "-shared", "-fPIC", "-x", "c++", "-", "-lcuda", "-o", str(work / "probe.so")],
                            input='extern "C" int q27x_probe() { return 0; }\n', text=True, capture_output=True,
                            timeout=120, env=os.environ.copy())
        info["probe"] = "PASS" if cp.returncode == 0 else "FAIL"
        if cp.returncode != 0:
            info["probe_stderr"] = (cp.stderr or "")[-1500:]
    except Exception as e:
        info["reason"] = f"{type(e).__name__}: {e}"[:500]
    return info

os.environ.setdefault("MAX_JOBS", "4")              # bound FlashInfer JIT build parallelism


def e19_event(msg):
    line = f"[{time.perf_counter() - NOTEBOOK_T0:7.0f}s] {msg}"
    print(line, flush=True)
    with E19_LOCK:
        E19_STATE["events"].append(line[:600])


def e19_save_state():
    """Atomic (tmp + fsync + replace): a crash mid-write leaves the previous state file. One writer at a time: the
    churn / long-stage worker threads save concurrently."""
    with E19_SAVE_LOCK:
        with E19_LOCK:
            snap = json.dumps(E19_STATE, default=str, indent=1)
        tmp = Path(str(E19_STATE_PATH) + ".tmp")
        with open(tmp, "w", encoding="utf-8") as f:
            f.write(snap)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp, E19_STATE_PATH)


def e19_write_row(row):
    """Append one raw row (flush + fsync) as soon as its request ends: a crash keeps every finished probe."""
    line = json.dumps(row, separators=(",", ":"), default=str)
    with E19_LOCK:
        with open(E19_ROWS_PATH, "a", encoding="utf-8") as f:
            f.write(line + "\n")
            f.flush()
            os.fsync(f.fileno())
        a = E19_STATE["arms"].get(row.get("arm"))
        if a is not None:
            a.setdefault("row_status", {})
            a["row_status"][row["status"]] = a["row_status"].get(row["status"], 0) + 1
    e19_save_state()


def e19_flags(arm, backend, max_len=None):
    """S1's ENGINE_CONFIG (CELL 5) as api_server flags; the arm sets MTP, prefix caching, the KV dtype and (arm E)
    async scheduling. Async scheduling is left at vLLM's default on every other arm, as in S1."""
    cfg = E19_ARMS[arm]
    f = ["--model", str(MODEL_PATH), "--tensor-parallel-size", "4", "--distributed-executor-backend", "mp",
         "--dtype", "auto", "--load-format", "safetensors", "--language-model-only",
         "--max-model-len", str(max_len or MAX_MODEL_LEN), "--max-num-seqs", str(MAX_NUM_SEQS),
         "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS), "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
         "--disable-custom-all-reduce", "--seed", str(SEED), "--attention-backend", backend,
         "--kv-cache-dtype", cfg["kv"]]
    f += ["--enable-prefix-caching"] if cfg["prefix"] else ["--no-enable-prefix-caching"]
    if cfg["spec"]:
        f += ["--speculative-config", json.dumps({"method": "mtp", "num_speculative_tokens": int(cfg["spec"])})]
    if cfg.get("async") is False:
        f += ["--no-async-scheduling"]           # arm E: the cheapest workaround for the async-scheduling race
    f += ["--host", "127.0.0.1", "--port", str(E19_PORT), "--served-model-name", "q27", "--enable-prompt-tokens-details"]
    return f


class E19Server:
    def __init__(self, name, flags):
        self.name, self.flags = name, list(flags)
        self.cmd = ([sys.executable, E19_FAKE] if E19_FAKE else
                    [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]) + self.flags
        self.env = dict(os.environ, VLLM_SERVER_DEV_MODE="1", PYTHONUNBUFFERED="1")
        self.log_path = E19_DIR / f"vllm_server_{name}.log"
        self.proc = None

    def start(self):
        self.log_f = open(self.log_path, "w", encoding="utf-8")
        self.t0 = time.perf_counter()
        self.proc = subprocess.Popen(self.cmd, env=self.env, stdout=self.log_f, stderr=subprocess.STDOUT,
                                     stdin=subprocess.DEVNULL, start_new_session=True, cwd=None if E19_FAKE else "/tmp")

    def alive(self):
        return self.proc is not None and self.proc.poll() is None

    def log_text(self):
        try:
            return self.log_path.read_text(encoding="utf-8", errors="replace")
        except Exception:
            return ""

    def wait_ready(self, timeout, silence=E19_SILENCE_SEC):
        """ready = /health answers AND this server's own 'GPU KV cache size' line is in its log (E18 rev 2); a log that
        stops growing for `silence` seconds is a stall."""
        last, last_t = -1, time.perf_counter()
        while time.perf_counter() - self.t0 < timeout:
            if not self.alive():
                return False, f"exited rc={self.proc.returncode}"
            try:
                _urlreq.urlopen(E19_URL + "/health", timeout=10).read()
                if "GPU KV cache size" in self.log_text():
                    return True, f"ready in {time.perf_counter() - self.t0:.0f}s"
            except Exception:
                pass
            sz = self.log_path.stat().st_size if self.log_path.exists() else 0
            if sz != last:
                last, last_t = sz, time.perf_counter()
            elif time.perf_counter() - last_t > silence:
                return False, f"stall ({silence:.0f} s of log silence)"
            time.sleep(0.5 if E19_FAKE else 5)
        return False, "timeout"

    def kill(self):
        """SIGTERM to the process group, a grace, then SIGKILL to the group ALWAYS (engine core / TP workers stuck in
        CUDA or NCCL can outlive the API server), then wait."""
        if self.proc is None:
            return
        grp = hasattr(os, "killpg")
        try:
            os.killpg(self.proc.pid, _signal.SIGTERM) if grp else self.proc.terminate()
        except Exception:
            pass
        try:
            self.proc.wait(timeout=20)
        except Exception:
            pass
        try:
            os.killpg(self.proc.pid, getattr(_signal, "SIGKILL", _signal.SIGTERM)) if grp else self.proc.kill()
        except Exception:
            pass
        try:
            self.proc.wait(timeout=30)
        except Exception:
            pass
        try:
            self.log_f.close()
        except Exception:
            pass


def e19_kill_gpu_pids():
    """Kill every process still holding a GPU compute context (never this kernel)."""
    if E19_FAKE:
        return []
    killed = []
    try:
        out = subprocess.run(["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"], capture_output=True,
                             text=True, timeout=20).stdout
        for tok in out.split():
            if tok.isdigit() and int(tok) != os.getpid():
                try:
                    os.kill(int(tok), getattr(_signal, "SIGKILL", _signal.SIGTERM))
                    killed.append(int(tok))
                except Exception:
                    pass
    except Exception:
        pass
    return killed


def e19_wait_gpu_free(max_sec=float(os.environ.get("Q27X_GPU_WAIT_MAX", 180))):
    """S1's rule: every GPU < 1 GB used (nvidia-smi)."""
    if E19_FAKE:
        return {"released": True, "sec": 0.0, "fake": True}
    w0, used = time.perf_counter(), []
    while time.perf_counter() - w0 < max_sec:
        try:
            used = [int(x) for x in subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                                   capture_output=True, text=True, timeout=20).stdout.split()]
        except Exception:
            used = []
        if used and max(used) < 1024:
            return {"released": True, "sec": round(time.perf_counter() - w0, 1), "last_used_mb": used}
        time.sleep(5)
    return {"released": False, "sec": round(time.perf_counter() - w0, 1), "last_used_mb": used}


def e19_stop_server():
    global E19_SERVER
    rec = {}
    if E19_SERVER is not None:
        E19_SERVER.kill()
        E19_SERVER = None
    rec["killed_gpu_pids"] = e19_kill_gpu_pids()
    rec["gpu"] = e19_wait_gpu_free()
    return rec


def e19_prom():
    txt = _urlreq.urlopen(E19_URL + "/metrics", timeout=15).read().decode("utf-8", "replace")
    out = {}
    for line in txt.splitlines():
        if not line or line.startswith("#"):
            continue
        m = re.match(r"^([A-Za-z_:][A-Za-z0-9_:]*)(\{[^}]*\})?\s+(\S+)", line)
        if m and not m.group(1).endswith("_created"):
            try:
                out[m.group(1)] = out.get(m.group(1), 0.0) + float(m.group(3))
            except ValueError:
                pass
    return out


E19_METRICS = {"gen": "vllm:generation_tokens_total", "prompt": "vllm:prompt_tokens_total",
               "pc_queries": "vllm:prefix_cache_queries_total", "pc_hits": "vllm:prefix_cache_hits_total",
               "preempt": "vllm:num_preemptions_total", "drafts": "vllm:spec_decode_num_drafts_total",
               "draft_tokens": "vllm:spec_decode_num_draft_tokens_total",
               "accepted": "vllm:spec_decode_num_accepted_tokens_total", "running": "vllm:num_requests_running"}


def e19_metrics():
    try:
        p = e19_prom()
        return {k: p.get(v) for k, v in E19_METRICS.items()}
    except Exception as e:
        return {"error": repr(e)[:200]}


def e19_delta(a, b):
    return {k: (round(b[k] - a[k], 1) if isinstance(a.get(k), (int, float)) and isinstance(b.get(k), (int, float)) else None)
            for k in E19_METRICS}


def e19_reset_cache():
    """POST /reset_prefix_cache (VLLM_SERVER_DEV_MODE=1). Called on every arm (a no-op with prefix caching off).
    vLLM does not report whether it worked; the cold rows' cached_tokens show it."""
    try:
        req = _urlreq.Request(E19_URL + "/reset_prefix_cache", data=b"", method="POST")
        return _urlreq.urlopen(req, timeout=60).status
    except Exception as e:
        return repr(e)[:120]


class E19Req(threading.Thread):
    """One greedy streaming /v1/completions request with prompt token ids. Records integer token ids, top-1 / top-2
    margins, positions where the emitted token is not the top-1 of its own logprobs, finish_reason, [DONE], the usage
    chunk's cached tokens and TTFT. stop() closes the stream (vLLM aborts the request on disconnect).
    The halt flag is _halt, NOT _stop (Thread._stop is a method join() calls on Python 3.12)."""
    def __init__(self, ids, max_new):
        super().__init__(daemon=True)
        self.body = {"model": "q27", "prompt": list(ids), "max_tokens": int(max_new), "temperature": 0.0, "top_p": 1.0,
                     "stream": True, "stream_options": {"include_usage": True}, "return_token_ids": True,
                     "logprobs": 2, "return_tokens_as_token_ids": True, "skip_special_tokens": False,
                     "ignore_eos": False}
        self.tokens, self.margins, self.mismatch = [], [], []
        self.err, self._resp, self._halt = None, None, False
        self.done_seen, self.finish_reason, self.usage = False, None, None
        self.t_start, self.t_first, self.t_end = None, None, None

    def run(self):
        self.t_start = time.perf_counter()
        try:
            req = _urlreq.Request(E19_URL + "/v1/completions", data=json.dumps(self.body).encode(), method="POST",
                                  headers={"Content-Type": "application/json"})
            self._resp = _urlreq.urlopen(req, timeout=900)
            for raw in self._resp:
                if self._halt:
                    break
                line = raw.strip()
                if not line.startswith(b"data:"):
                    continue
                payload = line[5:].strip()
                if payload == b"[DONE]":
                    self.done_seen = True
                    break
                d = json.loads(payload)
                if "error" in d or d.get("object") == "error":
                    self.err = ("stream error: " + json.dumps(d.get("error") or d.get("message") or d))[:300]
                    break
                if d.get("usage") is not None and not d.get("choices"):
                    self.usage = d["usage"]
                for c in d.get("choices") or []:
                    ids_ = c.get("token_ids")
                    lp = c.get("logprobs") or {}
                    if ids_ is None:
                        if c.get("text"):
                            self.err = "MISSING_STREAM_TOKEN_IDS"
                        ids_ = []
                    if any(not isinstance(v, int) or isinstance(v, bool) for v in ids_):
                        self.err = "INVALID_STREAM_TOKEN_IDS"
                        break
                    tops = lp.get("top_logprobs") or []
                    for j, t in enumerate(ids_):
                        top = tops[j] if j < len(tops) else None
                        m, mm = None, False
                        if isinstance(top, dict) and top:
                            items = []
                            for k_, v_ in top.items():
                                try:
                                    items.append((int(str(k_).split(":")[-1]), float(v_)))
                                except ValueError:
                                    pass
                            items.sort(key=lambda x: -x[1])
                            if len(items) >= 2:
                                m = round(items[0][1] - items[1][1], 4)
                            own = next((v_ for k_, v_ in items if k_ == t), None)
                            mm = bool(items) and (own is None or items[0][1] - own > 1e-4)
                        if mm:
                            self.mismatch.append(len(self.tokens) + j)
                        self.margins.append(m)
                    if ids_ and self.t_first is None:
                        self.t_first = time.perf_counter()
                    self.tokens.extend(ids_)
                    if c.get("finish_reason") is not None:
                        self.finish_reason = c["finish_reason"]
                if self.err:
                    break
        except Exception as e:
            if not self._halt:
                self.err = repr(e)[:300]
        self.t_end = time.perf_counter()
        if not self._halt and not self.err and not (self.done_seen and self.finish_reason):
            self.err = "INCOMPLETE_SSE: [DONE] seen %s, finish_reason %s" % (self.done_seen, self.finish_reason)

    def stop(self):
        self._halt = True
        try:
            self._resp.fp.raw._sock.shutdown(socket.SHUT_RDWR)
        except Exception:
            pass
        try:
            if self._resp is not None:
                self._resp.close()
        except Exception:
            pass


def e19_run_req(ids, max_new, deadline):
    """Run one request to completion or until `deadline` (then it is stopped: status not_tested_budget)."""
    g = E19Req(ids, max_new)
    g.start()
    while g.is_alive():
        g.join(min(2.0, max(0.05, deadline - time.perf_counter())))
        if time.perf_counter() >= deadline and g.is_alive():
            g.stop()
            g.join(10)
            g.err = g.err or "stopped: arm deadline"
            g.budget_stop = True
            break
        if E19_SERVER is not None and not E19_SERVER.alive() and g.is_alive():
            g.join(5)
            if g.is_alive():
                g.stop()
                g.join(10)
            g.err = g.err or "server process died"
            break
    return g


def e19_decode(ids):
    try:
        return tokenizer.decode(list(ids), skip_special_tokens=False)
    except Exception as e:
        return f"<decode error {e!r}>"[:200]


def e19_row(arm, stage, ps, idx, ids, g, conc, extra=None):
    usage = g.usage or {}
    det = usage.get("prompt_tokens_details") or {}
    cached = det.get("cached_tokens") if isinstance(det, dict) else None
    if g.usage is not None and cached is None:
        cached = 0           # vLLM 0.19 omits prompt_tokens_details when 0 tokens were cached (exclude_none)
    if not g.err:
        status = "ok"
    elif getattr(g, "budget_stop", False):
        status = "not_tested_budget"
    elif E19_SERVER is not None and not E19_SERVER.alive():
        status = "server_dead"
    else:
        status = "error"
    r = {"arm": arm, "stage": stage, "pass": ps, "idx": idx, "task": E19_PROMPTS[idx]["task"], "status": status,
         "error": g.err, "prompt_len": len(ids), "prompt_sha": hashlib.sha256(json.dumps(list(ids)).encode()).hexdigest()[:16],
         "max_tokens": g.body["max_tokens"], "n_tokens": len(g.tokens), "tokens": g.tokens, "margins": g.margins,
         "top1_mismatch": g.mismatch, "finish_reason": g.finish_reason, "done_seen": g.done_seen,
         "usage_seen": g.usage is not None, "prompt_tokens": usage.get("prompt_tokens"), "cached_tokens": cached,
         "usage": usage,
         "ttft": round(g.t_first - g.t_start, 3) if g.t_first and g.t_start else None,
         "sec": round((g.t_end or time.perf_counter()) - (g.t_start or time.perf_counter()), 2),
         "t_start": round((g.t_start or 0) - NOTEBOOK_T0, 1), "conc": conc, "text": e19_decode(g.tokens)}
    r.update(extra or {})
    return r


def e19_skip_row(arm, stage, ps, idx, status, why, extra=None):
    r = {"arm": arm, "stage": stage, "pass": ps, "idx": idx, "task": E19_PROMPTS[idx]["task"], "status": status,
         "error": why, "tokens": [], "margins": [], "top1_mismatch": [], "n_tokens": 0}
    r.update(extra or {})
    return r


In [ ]:
# ==================================================================== CELL E19-2 - prompts, arms, probes, verdicts, report
# Prompts: the 12 tasks of the profile, iteration 0 / expert 0, rendered by S1's own e10_render_request (verbatim below:
# poetiq_build_message + the chat template with enable_thinking and reasoning_effort xhigh), tokenized as S1's e10_admit
# does. The token ids are identical on every arm and every account.
# Per arm (one vLLM 0.19 server), in this order:
#   serial  - 6 prompts (every other one by length), one at a time: reset -> cold (greedy 1,024) -> warm (the same prompt
#             again: prefix-cache hit) -> [prefix ON: reset -> cold2 (A/A)]
#   churn   - the batch-reordering probe (suspect: the async-scheduling race, vLLM #53919, num_accepted_tokens reordered
#             on every row move): the 6 serial prompts x 4 = 24 requests, up to 14 at once, max_tokens staggered
#             256..2,048, arrivals staggered and refilled as requests finish, so rows join / finish / move continuously;
#             each row is compared with the same prompt run alone (serial cold, or serial warm when it was a cache hit)
#   long8k  - the other 6 prompts / long30k - the 6 serial prompts, 8 at once: a phase-2-style context. natural = prompt +
#             a deterministic filler (the task's own grids written out inside <think>, identical ids on every arm) up to
#             256 tokens before a 1,600-token block boundary, greedy 1,024; cont = prompt + filler + the first 512 tokens
#             the server generated itself (the cached prefix holds generated tokens), greedy 512; cont must reproduce
#             natural's tokens 512-1,024.
#   /metrics every E19_SAMPLE_SEC: MTP acceptance per window (accepted / draft tokens), preemptions, running.
# Every row is saved as it ends (q27x/e19/e19_rows.jsonl); verdicts: e19_verdicts() (E19-J).
E19_PROMPTS = []
E19_VERDICTS = {}
E19_SAMPLE_SEC = float(os.environ.get("E19_SAMPLE_SEC", 10))

# S1 solver cell, verbatim:
def e10_render_request(st, max_feedback=None):
    it=st['iteration']
    idx = POETIQ_SOLVER_VARIANTS[st['expert'] % len(POETIQ_SOLVER_VARIANTS)]
    limit = POETIQ_MAX_SOLUTIONS if max_feedback is None else max_feedback
    msg = poetiq_build_message(CHALLENGES[st['task_id']],it,st['expert'],st['history'],
           variant=idx,selection_probability=POETIQ_SELECTION_PROB,
           max_solutions=limit,improving_order=POETIQ_IMPROVING_ORDER,
           shuffle=POETIQ_SHUFFLE_EXAMPLES,seed=POETIQ_SEED+st['task_index']*1000+st['expert']*100)
    chat=[{'role':'user','content':msg}]
    return tokenizer.apply_chat_template(chat,tokenize=False,add_generation_prompt=True,
                        enable_thinking=True,reasoning_effort=REASONING_EFFORT)


def e19_filler_text(task, cycle):
    """Deterministic 'thinking' over the task's own training grids (identical on every arm)."""
    out = ["Let me carefully analyze the training examples.\n\n" if cycle == 0 else
           f"Let me re-check every example once more (pass {cycle + 1}).\n\n"]
    for k, ex in enumerate(task["train"]):
        gi, go = ex["input"], ex["output"]
        out.append(f"Example {k + 1} input ({len(gi)}x{len(gi[0])}):\n" + "\n".join(" ".join(map(str, r)) for r in gi) + "\n")
        out.append(f"Example {k + 1} output ({len(go)}x{len(go[0])}):\n" + "\n".join(" ".join(map(str, r)) for r in go) + "\n")
        if len(gi) == len(go) and len(gi[0]) == len(go[0]):
            ch = [(r, c, gi[r][c], go[r][c]) for r in range(len(gi)) for c in range(len(gi[0])) if gi[r][c] != go[r][c]]
            out.append(f"Comparing example {k + 1}: {len(ch)} cells change"
                       + ("".join(f", ({r},{c}) {a}->{b}" for r, c, a, b in ch[:24])) + ".\n\n")
        else:
            out.append(f"Comparing example {k + 1}: the size changes from {len(gi)}x{len(gi[0])} to {len(go)}x{len(go[0])}.\n\n")
    return "".join(out)


def e19_filler_ids(task, n):
    ids, cycle = [], 0
    while len(ids) < n:
        ids += tokenizer.encode(e19_filler_text(task, cycle), add_special_tokens=False)
        cycle += 1
    return ids[:n]


def e19_build_prompts():
    global E19_SERIAL_IDX, E19_LONG, E19_CHURN_JOBS
    for k, tid in enumerate(TASK_IDS):
        st = {"task_id": tid, "expert": 0, "task_index": k, "iteration": 0, "history": []}
        ids = tokenizer.encode(e10_render_request(st), add_special_tokens=False)
        E19_PROMPTS.append({"task": tid, "ids": ids, "len": len(ids)})
    by_len = sorted(range(len(E19_PROMPTS)), key=lambda i: (E19_PROMPTS[i]["len"], i))
    E19_SERIAL_IDX = sorted(by_len[::2][:E19_PLAN["n_serial"]])
    other = [i for i in range(len(E19_PROMPTS)) if i not in E19_SERIAL_IDX]
    lad, reps = E19_PLAN["churn_max_tokens"], E19_PLAN["churn_reps"]
    # job order interleaves prompts and lengths so that short and long requests alternate in the queue
    E19_CHURN_JOBS = [[i, r, lad[(k * reps + r * 3) % len(lad)]] for r in range(reps) for k, i in enumerate(E19_SERIAL_IDX)]
    bs, bo, gp = E19_PLAN["block"], E19_PLAN["backoff"], E19_PLAN["g_pre"]
    E19_LONG = []
    for name, n_ctx, which in E19_PLAN["long"]:
        idx = list(E19_SERIAL_IDX) if which == "serial" else other if which == "other" else list(range(len(E19_PROMPTS)))
        ctx = {}
        for i in idx:
            L = E19_PROMPTS[i]["len"]
            pf = -(-(L + n_ctx - gp) // bs) * bs - bo        # prompt + filler ends `backoff` tokens before a block boundary
            if pf - L < 1024 or pf + gp + E19_PLAN["cmp"] + 64 > MAX_MODEL_LEN:
                continue                                      # "where the context fits"
            ctx[i] = {"ids": E19_PROMPTS[i]["ids"] + e19_filler_ids(CHALLENGES[E19_PROMPTS[i]["task"]], pf - L), "pf_len": pf}
        E19_LONG.append({"name": name, "n_ctx": n_ctx, "idx": sorted(ctx), "ctx": ctx})
    E19_STATE["prompts"] = [{"task": p["task"], "len": p["len"],
                             "sha": hashlib.sha256(json.dumps(p["ids"]).encode()).hexdigest()[:16]} for p in E19_PROMPTS]
    E19_STATE["serial_idx"] = E19_SERIAL_IDX
    E19_STATE["churn_jobs"] = E19_CHURN_JOBS
    E19_STATE["long"] = [{"name": s["name"], "n_ctx": s["n_ctx"], "idx": s["idx"],
                          "pf_len": {str(i): s["ctx"][i]["pf_len"] for i in s["idx"]}} for s in E19_LONG]


def e19_meta(name):
    a = E19_STATE["arms"].get(name) or {}
    return {"arm": name, "prefix_caching": E19_ARMS[name]["prefix"], "spec": E19_ARMS[name]["spec"],
            "serial_idx": list(E19_SERIAL_IDX), "churn_jobs": [list(x) for x in E19_CHURN_JOBS],
            "long_stages": [{"name": s["name"], "idx": list(s["idx"])} for s in E19_LONG],
            "launch_failed": a.get("launch_failed"), "server_died": a.get("server_died"),
            "accept_windows": list(a.get("accept_windows") or [])}


def e19_parse_log(txt):
    g = lambda pat, f=str: (f(re.findall(pat, txt)[-1].replace(",", "")) if re.findall(pat, txt) else None)
    return {"version": g(r"vLLM API server version ([0-9][\w.+-]*)"),
            "kv_tokens": g(r"GPU KV cache size: ([\d,]+) tokens", int),
            "max_concurrency": g(r"Maximum concurrency for [\d,]+ tokens per request: ([\d.]+)x", float),
            "block_size": g(r"Setting attention block size to (\d+) tokens", int),
            "mamba_cache_mode": g(r"Mamba cache mode is set to '(\w+)'"),
            "async_lines": sorted(set(re.findall(r"[Aa]sync(?:hronous)? scheduling[^\n]{0,80}", txt)))[:6],
            "spec_lines": re.findall(r"SpecDecoding metrics:[^\n]{0,200}", txt)[-3:],
            "backend_lines": sorted(set(re.findall(r"(?:Using|Selected) [\w ]{0,40}?(?:backend|Backend)[^\n]{0,80}", txt)))[:12],
            "error_lines": [l[:300] for l in txt.splitlines() if re.search(r"\b(ERROR|Traceback|CUDA error|NaN)\b", l)][:30]}


def e19_conc_level(longest, want):
    """`want` streams, fewer when the KV pool cannot hold that many of the stage's longest requests (bf16 KV arm)."""
    a = E19_STATE["arms"][E19_CUR]
    cap = (a.get("log") or {}).get("max_concurrency")
    cap = cap * a.get("max_len", MAX_MODEL_LEN) if cap else (a.get("log") or {}).get("kv_tokens")
    if not cap:
        return want
    return int(max(1, min(want, (0.8 * cap) // max(1, longest))))


def e19_pool(jobs, level, deadline, stagger=0.0):
    """Run job(skip_status) callables on `level` worker threads (a worker takes the next job as soon as its request
    ends: refilled arrivals); worker k starts after k * stagger seconds. A job taken after the deadline or with a dead
    server is told to write skip rows instead of running."""
    q, lk = list(jobs), threading.Lock()

    def worker(k):
        time.sleep(k * stagger)
        while True:
            with lk:
                if not q:
                    return
                job = q.pop(0)
            if E19_SERVER is None or not E19_SERVER.alive():
                job("server_dead")
            elif time.perf_counter() > deadline - 5:
                job("not_tested_budget")
            else:
                job(None)
    th = [threading.Thread(target=worker, args=(k,), daemon=True) for k in range(max(1, level))]
    for t in th:
        t.start()
    for t in th:
        t.join()


class E19Sampler(threading.Thread):
    """/metrics every E19_SAMPLE_SEC while an arm runs: the per-window deltas show an MTP-acceptance collapse."""
    def __init__(self, arm):
        super().__init__(daemon=True)
        self.arm, self._halt, self.prev = arm, False, None

    def run(self):
        while not self._halt:
            m = e19_metrics()
            if "error" not in m:
                m["t"] = round(time.perf_counter() - NOTEBOOK_T0, 1)
                if self.prev is not None:
                    d = e19_delta(self.prev, m)
                    d.update(t=m["t"], running=m.get("running"))
                    with E19_LOCK:
                        E19_STATE["arms"][self.arm].setdefault("accept_windows", []).append(d)
                self.prev = m
            end = time.perf_counter() + E19_SAMPLE_SEC
            while not self._halt and time.perf_counter() < end:
                time.sleep(0.2)

    def stop(self):
        self._halt = True
        self.join(15)


def e19_alive(name):
    if E19_SERVER is not None and E19_SERVER.alive():
        return True
    a = E19_STATE["arms"][name]
    if not a.get("server_died"):
        a["server_died"] = "server process exited rc=%s at %.0fs; log tail: %s" % (
            E19_SERVER.proc.returncode if E19_SERVER is not None and E19_SERVER.proc else None,
            time.perf_counter() - NOTEBOOK_T0, (E19_SERVER.log_text()[-1500:] if E19_SERVER is not None else ""))
        e19_event(f"arm {name}: SERVER DIED")
    return False


def e19_stage_serial(name, deadline):
    pc = E19_ARMS[name]["prefix"]
    for i in E19_SERIAL_IDX:
        if not e19_alive(name):
            return
        if deadline - time.perf_counter() < E19_MIN_SERIAL_SEC:
            e19_event(f"arm {name}: serial stage stopped by the budget before prompt {i}")
            return
        ids = E19_PROMPTS[i]["ids"]
        for ps in ["cold", "warm"] + (["cold2"] if pc else []):
            if not e19_alive(name):
                return
            rs = e19_reset_cache() if ps != "warm" else None
            m0 = e19_metrics()
            g = e19_run_req(ids, E19_PLAN["max_new"], deadline)
            e19_write_row(e19_row(name, "serial", ps, i, ids, g, 1,
                                  {"reset": rs, "metric_delta": e19_delta(m0, e19_metrics())}))


def e19_stage_churn(name, deadline):
    if not e19_alive(name):
        return
    if deadline - time.perf_counter() < E19_MIN_CHURN_SEC:
        e19_event(f"arm {name}: churn stage not started (budget)")
        return
    lvl = e19_conc_level(max(p["len"] for p in E19_PROMPTS) + max(E19_PLAN["churn_max_tokens"]), E19_PLAN["churn_conc"])
    rs = e19_reset_cache()
    m0, t0 = e19_metrics(), time.perf_counter()

    def mk(i, rep, mx):
        def job(skip):
            ex = {"reset": rs, "rep": rep}
            if skip:
                e19_write_row(e19_skip_row(name, "churn", f"j{rep}", i, skip, skip, ex))
                return
            g = e19_run_req(E19_PROMPTS[i]["ids"], mx, deadline)
            e19_write_row(e19_row(name, "churn", f"j{rep}", i, E19_PROMPTS[i]["ids"], g, lvl, ex))
        return job
    e19_pool([mk(*j) for j in E19_CHURN_JOBS], lvl, deadline, stagger=E19_PLAN["churn_stagger_sec"])
    E19_STATE["arms"][name]["stages"]["churn"] = {"level": lvl, "sec": round(time.perf_counter() - t0, 1),
                                                 "metric_delta": e19_delta(m0, e19_metrics())}
    e19_save_state()


def e19_stage_long(name, st, deadline):
    if not e19_alive(name) or not st["idx"]:
        return
    if deadline - time.perf_counter() < E19_MIN_LONG_SEC:
        e19_event(f"arm {name}: {st['name']} not started (budget)")
        return
    gp, cm = E19_PLAN["g_pre"], E19_PLAN["cmp"]
    lvl = e19_conc_level(max(st["ctx"][i]["pf_len"] for i in st["idx"]) + gp + cm, E19_PLAN["long_conc"])
    rs = e19_reset_cache()
    m0, t0 = e19_metrics(), time.perf_counter()

    def mk(i):
        c = st["ctx"][i]
        ex = {"pf_len": c["pf_len"], "g_pre": gp, "n_ctx": st["n_ctx"], "reset": rs}

        def job(skip):
            if skip:
                e19_write_row(e19_skip_row(name, st["name"], "nat", i, skip, skip, ex))
                e19_write_row(e19_skip_row(name, st["name"], "cont", i, skip, skip, ex))
                return
            g = e19_run_req(c["ids"], gp + cm, deadline)
            r = e19_row(name, st["name"], "nat", i, c["ids"], g, lvl, ex)
            e19_write_row(r)
            if r["status"] != "ok":
                e19_write_row(e19_skip_row(name, st["name"], "cont", i, r["status"], "natural row " + r["status"], ex))
                return
            if len(g.tokens) < gp + 16:
                e19_write_row(e19_skip_row(name, st["name"], "cont", i, "na_short",
                                           f"natural stopped after {len(g.tokens)} tokens", ex))
                return
            ids2 = list(c["ids"]) + list(g.tokens[:gp])
            if not e19_alive(name):
                e19_write_row(e19_skip_row(name, st["name"], "cont", i, "server_dead", "server dead", ex))
                return
            g2 = e19_run_req(ids2, cm, deadline)
            e19_write_row(e19_row(name, st["name"], "cont", i, ids2, g2, lvl, ex))
        return job
    e19_pool([mk(i) for i in st["idx"]], lvl, deadline)
    E19_STATE["arms"][name]["stages"][st["name"]] = {"level": lvl, "sec": round(time.perf_counter() - t0, 1),
                                                    "metric_delta": e19_delta(m0, e19_metrics())}
    e19_save_state()


def e19_fill_missing(name):
    """Every planned row not written gets a row: server_dead after a death, else not_tested_budget."""
    done = set()
    if E19_ROWS_PATH.exists():
        for r in e19_load_rows(E19_ROWS_PATH):
            if r.get("arm") == name:
                done.add(e19_key(r))
    a = E19_STATE["arms"][name]
    st = "server_dead" if a.get("server_died") else "not_tested_budget"
    for k in e19_planned_keys(e19_meta(name)):
        if k not in done:
            e19_write_row(e19_skip_row(name, k[0], k[1], k[2], st, a.get("not_launched") or st))


def e19_launch(name, deadline):
    global E19_SERVER
    a = E19_STATE["arms"][name]
    backend = "FLASHINFER" if CUDA_LINK.get("probe") == "PASS" else "TRITON_ATTN"     # S1 CELL 5's rule
    for max_len in [None] + ([65536] if E19_ARMS[name]["kv"] == "auto" else []):
        flags = e19_flags(name, backend, max_len)
        a.update(flags=flags, backend=backend, max_len=max_len or MAX_MODEL_LEN)
        srv = E19Server(name if max_len is None else f"{name}_len{max_len}", flags)
        e19_event(f"arm {name}: launching ({shlex.join(flags)[:400]})")
        srv.start()
        try:
            ok, why = srv.wait_ready(max(60.0, min(E19_READY_MAX_SEC, deadline - time.perf_counter())))
        except BaseException:
            srv.kill()
            raise
        txt = srv.log_text()
        a["launch"] = {"ok": ok, "why": why, "sec": round(time.perf_counter() - srv.t0, 1)}
        a["log"] = e19_parse_log(txt)
        if ok:
            E19_SERVER = srv
            return True
        a["launch_tail"] = txt[-4000:]
        srv.kill()
        a["gpu_after_failed_launch"] = {"killed": e19_kill_gpu_pids(), "gpu": e19_wait_gpu_free()}
        # bf16 KV ("lower context if needed"): retry once with 65,536 only when the KV pool cannot hold 106,496
        if not (max_len is None and E19_ARMS[name]["kv"] == "auto"
                and re.search(r"(KV cache|max_model_len|max seq len|maximum number of tokens)", txt[-6000:], re.I)
                and re.search(r"(larger than|not enough|insufficient)", txt[-6000:], re.I)):
            break
    a["launch_failed"] = f"server launch failed: {why}"
    return False


def e19_arm(name, deadline):
    global E19_CUR
    E19_CUR = name
    a = E19_STATE["arms"][name] = {"arm": name, "cfg": E19_ARMS[name], "stages": {},
                                   "deadline_rel": round(deadline - NOTEBOOK_T0, 1),
                                   "start_rel": round(time.perf_counter() - NOTEBOOK_T0, 1)}
    E19_STATE["order"].append(name)
    e19_save_state()
    sampler = None
    try:
        if deadline - time.perf_counter() < E19_LAUNCH_MIN_SEC:
            a["not_launched"] = "NOT_TESTED_BUDGET: %.0f s left before the arm deadline" % (deadline - time.perf_counter())
            e19_event(f"arm {name}: {a['not_launched']}")
            return
        if not e19_launch(name, deadline):
            e19_event(f"arm {name}: LAUNCH FAILED: {a['launch_failed']}")
            return
        e19_event(f"arm {name}: {a['launch']['why']} | KV {a['log'].get('kv_tokens')} | block {a['log'].get('block_size')}"
                  f" | max concurrency {a['log'].get('max_concurrency')} | version {a['log'].get('version')}"
                  f" | async {a['log'].get('async_lines')}")
        # rev 2 (GPT audit 2 HIGH-4): the async A/B is only valid when the server really runs the intended mode
        _al = " ".join(a["log"].get("async_lines") or [])
        a["effective_async"] = ("enabled" if re.search(r"scheduling is\s+enabled", _al, re.I) else
                                "disabled" if re.search(r"scheduling is\s+disabled", _al, re.I) else "unknown")
        _want = "disabled" if E19_ARMS[name].get("async") is False else "enabled"
        if name in ("D", "E") and a["effective_async"] != _want and not (_want == "disabled" and a["effective_async"] == "unknown"):
            a["launch_failed"] = f"async mode mismatch: expected {_want}, observed {a['effective_async']} ({_al[:160]})"
            e19_event(f"arm {name}: LAUNCH FAILED: {a['launch_failed']}")
            return
        sm_ids = tokenizer.encode("Reply with the digits 1 2 3 4 5 6 7 8:", add_special_tokens=False)
        g = e19_run_req(sm_ids, 16, deadline)          # S1's engine smoke, through the server
        a["smoke"] = {"tokens": len(g.tokens), "error": g.err, "margins_present": any(m is not None for m in g.margins),
                      "usage_seen": g.usage is not None, "text": e19_decode(g.tokens)[:120]}
        if not g.tokens:
            a["launch_failed"] = f"engine smoke produced no tokens: {g.err}"
            e19_event(f"arm {name}: SMOKE FAILED {g.err}")
            return
        e19_save_state()
        sampler = E19Sampler(name)
        sampler.start()
        t0 = time.perf_counter()
        m0 = e19_metrics()
        e19_stage_serial(name, deadline)
        a["stages"]["serial"] = {"sec": round(time.perf_counter() - t0, 1), "metric_delta": e19_delta(m0, e19_metrics())}
        e19_stage_churn(name, deadline)
        for st in E19_LONG:
            e19_stage_long(name, st, deadline)
        e19_alive(name)
        a["log_end"] = e19_parse_log(E19_SERVER.log_text()) if E19_SERVER is not None else None
    except Exception as e:
        a["error"] = f"{type(e).__name__}: {e}"[:600]
        a["traceback"] = traceback.format_exc()[-3000:]
        e19_event(f"arm {name}: CRASH {a['error']}")
    finally:
        if sampler is not None:
            sampler.stop()
        try:
            e19_fill_missing(name)
        except Exception as e:
            a["fill_error"] = repr(e)[:300]
        a["stop"] = e19_stop_server()
        a["end_rel"] = round(time.perf_counter() - NOTEBOOK_T0, 1)
        a["meta"] = e19_meta(name)
        e19_save_state()
        e19_event(f"arm {name}: done ({a['end_rel'] - a['start_rel']:.0f} s); GPU released {a['stop'].get('gpu', {}).get('released')}")


def e19_report():
    rows = e19_load_rows(E19_ROWS_PATH) if E19_ROWS_PATH.exists() else []
    metas = {n: (E19_STATE["arms"][n].get("meta") or e19_meta(n)) for n in E19_STATE["order"]}
    E19_VERDICTS.update(e19_verdicts(rows, metas))
    E19_STATE["verdicts"] = E19_VERDICTS
    e19_save_state()
    L = e19_report_lines(E19_VERDICTS, E19_ARMS, E19_STATE["arms"],
                         f"# E19 vLLM MTP / prefix-cache correctness: profile {PROFILE}, run {RUN_ID}")
    L += ["", "Calibrators in this run: " + (", ".join(a for a in ("A", "B") if a in metas) or
                                             "none (run src/e19_compare.py over the three accounts' bundles)"), "",
          "## Stages", ""]
    for name in E19_STATE["order"]:
        a = E19_STATE["arms"][name]
        L.append(f"- {name}: smoke {a.get('smoke')}; stages " + json.dumps(a.get("stages"), default=str)[:1500])
    (E19_DIR / "e19_report.md").write_text("\n".join(L) + "\n", encoding="utf-8")
    print("\n".join(L), flush=True)


if E19_ON:
    try:
        CUDA_LINK = ensure_libcuda_linkable()             # S1 CELL 5's -lcuda shim + link probe (FlashInfer JIT)
        CONFIG["cuda_link"] = CUDA_LINK
        E19_STATE["cuda_link"] = CUDA_LINK
        e19_event("CUDA/FlashInfer linker preflight: " + json.dumps(CUDA_LINK)[:300])
        e19_build_prompts()
        e19_event(f"prompts: {[p['len'] for p in E19_PROMPTS]} tokens; serial {E19_SERIAL_IDX}; churn "
                  f"{len(E19_CHURN_JOBS)} jobs; long " + str([(s['name'], len(s['idx'])) for s in E19_LONG]))
        _arms = [tuple(x) for x in P["e19_arms"]]
        E19_STATE["planned_arms"] = [x[0] for x in _arms]
        for _k, (_name, _reserve_min) in enumerate(_arms):
            _later = sum(r for _, r in _arms[_k + 1:]) * 60.0
            _deadline = HARD_STOP - E19_REPORT_RESERVE_SEC - _later
            e19_arm(_name, _deadline)
            if not (E19_STATE["arms"][_name].get("stop") or {}).get("gpu", {}).get("released", True):
                e19_event("GPU memory not released: no further arms")
                for _n2, _ in _arms[_k + 1:]:
                    E19_STATE["arms"][_n2] = {"arm": _n2, "cfg": E19_ARMS[_n2], "stages": {},
                                              "not_launched": "NOT_TESTED_BUDGET: GPU memory not released"}
                    E19_STATE["order"].append(_n2)
                    E19_STATE["arms"][_n2]["meta"] = e19_meta(_n2)
                break
    except Exception as _e:
        E19_STATE["fatal"] = f"{type(_e).__name__}: {_e}"[:800]
        E19_STATE["fatal_tb"] = traceback.format_exc()[-3000:]
        print("E19 FATAL:", E19_STATE["fatal_tb"], flush=True)
        try:
            e19_stop_server()
        except Exception:
            pass
    finally:
        try:
            e19_report()
        except Exception as _e:
            E19_STATE["report_error"] = traceback.format_exc()[-2000:]
            print("E19 report error:", E19_STATE["report_error"], flush=True)
        e19_save_state()
        try:
            with zipfile.ZipFile(WORKING / "Q27X_E19_BUNDLE.zip", "w", zipfile.ZIP_DEFLATED) as _z:
                for _p in sorted(E19_DIR.rglob("*")):
                    if _p.is_file():
                        _z.write(_p, _p.relative_to(OUT).as_posix())
            print("bundle:", WORKING / "Q27X_E19_BUNDLE.zip", flush=True)
        except Exception as _e:
            print("bundle error:", repr(_e)[:300], flush=True)
else:
    print("E19 not run:", "competition rerun (the CELL 0 identity submission stands)" if RERUN else f"profile {PROFILE} has no E19 arms",
          flush=True)
